# Portfolio note — historical student experiment

This notebook preserves the original modelling and analysis code, but all code-cell outputs have been cleared for public sharing. Selected original evaluation charts are provided separately in `assets/`.

**Data availability:** The `Pizza reviews.csv` coursework dataset and cached translations are not distributed in this repository. The notebook therefore cannot be executed end-to-end without authorised access to that dataset and required translation resources.

**Evaluation limitations:** The original notebook fits the tokenizer on all cleaned reviews **before** dividing the data into training and test sets. The recorded 91.03% accuracy and ~0.90 macro F1 are historical results, not a fully leakage-free, independently replicated benchmark. Multiple architectures/experiments were also compared using the held-out split. See `EVALUATION_NOTES.md` for details. The supplied model file is a historical artifact, not a standalone inference API (tokenizer/label encoder were not exported).


# PART B: Pizza Review Sentiment Analysis using Recurrent Neural Networks (RNN)

## Introduction

This project focuses on sentiment analysis of pizza reviews using Recurrent Neural Network (RNN) architectures. The main objective is to classify customer reviews into three sentiment categories:

- Negative
- Neutral
- Positive

The project involves several stages, including data cleaning, exploratory data analysis (EDA), feature engineering, text preprocessing, neural network modelling, model improvement, evaluation, error analysis, and final model justification.

The dataset contains reviews written in multiple languages, primarily English and Malay. Instead of discarding the Malay reviews, they are translated into English to preserve potentially useful training information and increase the amount of available textual data for model learning. This approach helps improve dataset utilisation while maintaining consistency in the language used for modelling.

In this workflow, tokenization and vocabulary construction are performed before the train-test split to ensure a consistent word index representation across the dataset. The resulting tokenized sequences are then padded to a fixed length before being used for deep learning model training.

---
# Background Research and Project Context

Sentiment analysis is a natural language processing task that aims to identify the emotional tone or opinion expressed in text. In this project, sentiment analysis is applied to pizza review data to classify customer opinions into three categories: Negative, Neutral, and Positive.

Customer reviews are valuable because they contain direct feedback about product quality, taste, texture, service experience, and overall satisfaction. For food businesses, analysing review sentiment can help identify common customer preferences, detect dissatisfaction early, and support data-driven decision-making.

This project uses recurrent neural network models because review text is sequential data, where word order and context can influence meaning. For example, phrases such as “not good” and “very good” contain similar words but express opposite sentiments depending on word order. Recurrent architectures such as SimpleRNN, LSTM, and GRU are therefore suitable for modelling textual sequences.

The dataset also contains reviews written in different languages. This creates an important preprocessing decision: whether to keep only English reviews or retain other language reviews after translation. Removing non-English reviews may simplify preprocessing, but it can reduce dataset size and discard useful sentiment information. Retaining translated reviews may improve training data coverage, but translation may also introduce semantic noise or reduce emotional nuance.

Therefore, this project investigates both modelling and preprocessing decisions through experiments. The workflow includes data cleaning, language handling, sentiment label engineering, review length analysis, tokenization, sequence padding, recurrent neural network modelling, model comparison, class weighting, and error analysis.

The final objective is to develop a well-justified deep learning sentiment classification model that can accurately classify multilingual pizza reviews while explaining the decisions made during preprocessing, model design, and evaluation.

## Why Sentiment Classification Is Used

The original dataset contains numerical review scores, which means both regression and classification are possible.

However, this project uses classification because the main objective is to understand customer sentiment rather than predict an exact numerical score. Classifying reviews into Negative, Neutral, and Positive categories provides clearer business insight and makes the results easier to interpret.

Sentiment classification is also suitable for evaluating model performance using classification metrics such as accuracy, precision, recall, F1-score, Macro F1-score, and confusion matrices.

This approach aligns with real-world customer feedback analysis, where businesses are often more interested in understanding whether a review reflects dissatisfaction, neutrality, or satisfaction rather than predicting the exact rating value.

## Step 1: Random Seed Setup

Random seeds are configured to improve reproducibility and experimental consistency throughout the project.

Deep learning models involve multiple sources of randomness, including weight initialization, data shuffling, dropout behaviour, and TensorFlow operations. Setting fixed random seeds helps ensure that model training results remain more stable and comparable across repeated runs.

This improves the reliability of model evaluation and allows fairer comparison between different neural network architectures and hyperparameter configurations.

Although random seed configuration improves reproducibility, slight variations may still occur due to GPU operations and backend optimisations in TensorFlow.

In [ ]:
import os
import random
import numpy as np
import tensorflow as tf

SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)  # the'___' stable hashing/order
random.seed(SEED) #python randomness
np.random.seed(SEED) #numpy randomness
tf.random.set_seed(SEED) #tensorflow randomness

## Step 2: Import Libraries

The required libraries are imported to support data preprocessing, exploratory data analysis (EDA), text processing, deep learning model development, visualisation, and model evaluation.

This project uses Python-based machine learning and deep learning libraries such as Pandas, NumPy, Scikit-learn, TensorFlow/Keras, Matplotlib, Seaborn, and NLTK to build and evaluate recurrent neural network architectures for multilingual sentiment classification.

The imported libraries serve different purposes throughout the workflow:

- **Pandas** and **NumPy** are used for data manipulation and numerical computation.
- **Matplotlib** and **Seaborn** are used for data visualisation and exploratory analysis.
- **Scikit-learn** provides utilities for data splitting, label encoding, evaluation metrics, and class weight computation.
- **TensorFlow/Keras** is used to develop and train deep learning models such as SimpleRNN, LSTM, and GRU architectures.
- **NLTK** is used for text preprocessing tasks such as tokenization and lemmatization.
- **GoogleTranslator** is used to translate Malay reviews into English to maintain language consistency across the dataset.

In [ ]:
# Uncomment if NLTK is not installed
# !pip install nltk

In [ ]:
import os
import re
import random

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from deep_translator import GoogleTranslator
import subprocess
import sys

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score
)
from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras.models import Sequential

from tensorflow.keras.layers import (
    Embedding,
    SimpleRNN,
    LSTM,
    GRU,
    Dense,
    Dropout
)

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint
)

# NLTK imports
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

import nltk

# Download required NLTK resources
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("wordnet")

## Step 3: Load Dataset

The pizza review dataset is loaded and inspected to understand its structure, dimensions, feature types, missing values, and sample records.

Initial inspection helps identify potential data quality issues such as incorrect data types, missing values, unnecessary columns, and multilingual review content before preprocessing and modelling.

In [ ]:
df = pd.read_csv("Pizza reviews.csv")

print("Dataset shape:", df.shape)
df.info()
df.head()

## Insights

The dataset contains 902 pizza review records with four columns:

- `Review`
- `Score`
- `Language`
- An additional column related to data generation suggestions

The `Review` column contains textual customer feedback, while the `Score` column represents numerical review ratings. However, the `Score` column is currently stored as an `object` data type instead of a numeric type, indicating that preprocessing and datatype conversion will be required before modelling.

The additional column related to data generation contains only 2 non-null values out of 902 records. Since the column is almost entirely empty and does not contribute meaningful information for sentiment analysis, it is likely to be removed during the data cleaning stage.

The dataset also contains multilingual reviews, primarily in English and Malay. Multilingual text introduces vocabulary inconsistencies and language-specific sentence structures, which may affect tokenization and overall model performance. Therefore, language handling and translation strategies will be explored carefully later in the workflow.

In addition, the dataset size is relatively small for deep learning applications. Smaller datasets increase the risk of overfitting, where the model memorises training patterns instead of learning generalisable sentiment representations. As a result, proper preprocessing, regularisation techniques, and model evaluation become especially important in this project.

Overall, the initial inspection provides a clearer understanding of the dataset structure and highlights several preprocessing challenges that must be addressed before neural network modelling can be performed effectively.

## Step 4: Data Cleaning

Data cleaning is performed to improve dataset quality and ensure that the data is suitable for sentiment classification modelling.

### 4.1 Handle Missing and Invalid Scores

The `Score` column is cleaned and converted into numeric format because review scores will later be used to generate sentiment labels for supervised learning.

Rows containing missing or invalid score values are removed to ensure that all remaining records contain reliable target information for model training and evaluation.

In [ ]:
# Reload original dataset
df = pd.read_csv("Pizza reviews.csv")

# Store original row count
original_rows = len(df)

# Convert Score column into numeric format
df["Score"] = pd.to_numeric(
    df["Score"],
    errors="coerce"
)

# Remove rows with missing or invalid scores
# Reset index to maintain sequential row ordering
df = df.dropna(subset=["Score"]).reset_index(drop=True)

# Display dataset shape
print("Dataset shape after cleaning:", df.shape)

# Display number of removed rows
print(
    "Rows removed:",
    original_rows - len(df)
)

# Display dataset information
print("\nDataset Information:")
df.info()

# Display missing values
print("\nMissing Values:")
print(df.isnull().sum())

## Insights

The `Score` column was successfully converted from `object` type into numerical format (`float64`), making it suitable for sentiment label generation and supervised learning tasks.

A total of 5 records containing missing or invalid score values were removed during the cleaning process. Since review scores are directly used to create sentiment labels, retaining incomplete or invalid target values could introduce noise and reduce the reliability of model training and evaluation.

After cleaning, the dataset contains 897 valid review records with complete numerical score information.

The additional column related to data generation suggestions now contains entirely missing values and no longer contributes meaningful information. This strongly indicates that the column is irrelevant for sentiment analysis and should be removed during subsequent preprocessing steps.

From a machine learning perspective, ensuring clean and valid target labels is particularly important because deep learning models are highly sensitive to noisy or inconsistent supervisory signals. Improper target values may negatively affect the model’s ability to learn meaningful sentiment patterns.

Overall, the dataset is now cleaner, more consistent, and more suitable for downstream preprocessing, sentiment labelling, and recurrent neural network modelling.

### 4.2 Remove Unnecessary Columns

Irrelevant and non-informative columns are removed to simplify the dataset and reduce unnecessary noise during preprocessing and modelling.

Columns containing excessive missing values or unrelated information are unlikely to contribute meaningful predictive value for sentiment classification. Removing such features helps streamline the workflow and improves overall dataset quality for downstream analysis.

In [ ]:
unnecessary_col = "Are there ways for you to generate more data? Spliting up sentences, would that help?"

if unnecessary_col in df.columns:
    df = df.drop(columns=[unnecessary_col])

df.columns

## Insights

The additional column related to data generation suggestions was removed because it contained entirely missing values and did not provide meaningful information for sentiment classification.

After removal, the dataset now contains only the core features required for the project:

- Review text (`Review`)
- Numerical review scores (`Score`)
- Language information (`Language`)

Removing irrelevant features simplifies the preprocessing pipeline, reduces unnecessary dataset noise, and improves workflow clarity for downstream sentiment analysis modelling.

From a machine learning perspective, retaining irrelevant or empty features may unnecessarily complicate preprocessing and increase the risk of introducing unhelpful patterns during modelling. Therefore, removing non-contributing columns helps create a cleaner and more focused dataset for recurrent neural network training.

### 4.3 Inspect Language Categories

The language distribution of the dataset is inspected to understand the multilingual composition of the reviews and determine which language categories are suitable for modelling.

Extremely small language categories may not provide sufficient samples for reliable learning and could introduce unnecessary noise during sentiment classification. Analysing the language distribution also helps guide decisions regarding translation, filtering, and preprocessing strategies for multilingual text data.

In [ ]:
# Display the number of reviews for each language category
print(df["Language"].value_counts())

## Insights

The dataset contains multiple language categories, with English and Malay forming the overwhelming majority of the reviews.

English reviews account for 491 records, while Malay reviews account for 402 records. In contrast, other language categories such as `Guga`, `Chinese`, and `Nippon` contain only 1–2 samples each.

Extremely small language categories are unlikely to provide sufficient data for reliable pattern learning, meaningful evaluation, or strong model generalisation. Retaining such categories may also introduce unnecessary vocabulary sparsity and noise into the NLP pipeline, especially when using tokenization-based deep learning models.

Additionally, categories such as `Guga` and `Nippon` may represent inconsistent or incorrectly labelled language entries, further reducing their usefulness for modelling.

Since English and Malay reviews represent almost the entire dataset, focusing on these dominant language groups allows the project to maintain sufficient training data while reducing unnecessary linguistic complexity.

Therefore, subsequent preprocessing and sentiment classification modelling will primarily focus on English and Malay reviews.

### 4.4 Retain English and Malay Reviews

Only the dominant English and Malay review categories are retained for subsequent preprocessing and sentiment modelling.

Removing extremely small language categories helps reduce vocabulary noise, improve dataset consistency, and ensure that the retained language groups contain sufficient samples for meaningful learning and evaluation.

In [ ]:
# Retain only English and Malay reviews
df = df[
    df["Language"].isin(
        ["English", "Malay"]
    )
].copy()

# Reset row indices after filtering
df = df.reset_index(drop=True)

# Display remaining language counts
print(df["Language"].value_counts())

# Display updated dataset shape
print("\nDataset Shape:", df.shape)

## Insights

Only English and Malay reviews were retained because they represent the overwhelming majority of the dataset, accounting for 893 out of 897 cleaned records.

The extremely small language categories (`Guga`, `Chinese`, and `Nippon`) were removed because they contain insufficient samples for reliable pattern learning, meaningful evaluation, or robust model generalisation.

After filtering, the dataset remains reasonably balanced between English and Malay reviews, which helps support more stable multilingual sentiment analysis and downstream translation-based preprocessing.

This filtering step preserves almost all available training data while reducing unnecessary vocabulary variability and improving overall dataset consistency. Reducing linguistic noise is particularly important for tokenization-based NLP models because rare or inconsistent language patterns may negatively affect vocabulary learning and sequence representation quality.

Overall, the dataset is now more focused, consistent, and suitable for recurrent neural network modelling.

### 4.5 Create Sentiment Labels

The numerical review scores are converted into categorical sentiment labels to transform the problem into a multiclass sentiment classification task.

The sentiment categories are defined using the following score ranges:

- Scores below 4 → Negative
- Scores from 4 to below 7 → Neutral
- Scores 7 and above → Positive

This label engineering process allows recurrent neural network models to learn and predict discrete sentiment categories directly from review text.

In [ ]:
# Function to convert numerical scores into sentiment categories
def get_sentiment(score):

    # Negative sentiment
    if score < 4:
        return "Negative"

    # Neutral sentiment
    elif score < 7:
        return "Neutral"

    # Positive sentiment
    else:
        return "Positive"


# Apply sentiment conversion to Score column
df["Sentiment"] = df["Score"].apply(
    get_sentiment
)

# Display sentiment class distribution
print(df["Sentiment"].value_counts())

## Insights

The numerical review scores were successfully transformed into three categorical sentiment labels:

- Negative
- Neutral
- Positive

This converts the original numerical scoring problem into a multiclass sentiment classification task suitable for recurrent neural network modelling.

The selected score thresholds help separate clearly negative reviews from moderately rated reviews and highly positive reviews, allowing the model to learn more meaningful sentiment distinctions from textual patterns.

The resulting sentiment distribution is reasonably balanced, with:

- 356 Positive reviews
- 291 Negative reviews
- 246 Neutral reviews

Although Positive reviews form the largest class, the class imbalance is not severe enough to heavily bias the dataset toward a single sentiment category. A relatively balanced class distribution is beneficial because it helps improve model stability, learning quality, and evaluation reliability.

Overall, the generated sentiment labels provide an appropriate target structure for supervised deep learning-based sentiment analysis.

In [ ]:
# Display percentage distribution of sentiment classes
print(
    df["Sentiment"]
    .value_counts(normalize=True) * 100
)

## Insights

The percentage distribution further confirms that the sentiment classes are relatively well distributed across the dataset:

- Positive: approximately 39.9%
- Negative: approximately 32.6%
- Neutral: approximately 27.5%

Although Positive reviews form the largest sentiment category, the differences between class proportions are not excessively large. This reduces the likelihood of the model becoming overly biased toward a single sentiment class during training.

Analysing percentage distributions is important because deep learning models trained on highly imbalanced datasets may achieve misleadingly high accuracy by overpredicting dominant classes while performing poorly on minority categories.

In this dataset, the sentiment proportions remain sufficiently balanced to support more reliable multiclass sentiment classification and fairer evaluation across all sentiment categories.

## Step 5: Malay-to-English Translation

Malay reviews are translated into English before text preprocessing and tokenization. This helps reduce vocabulary fragmentation because semantically similar words across different languages can be represented using a more consistent English vocabulary space.

Without translation, simple tokenization would treat English and Malay words as completely separate tokens even when they express similar meanings. Translating Malay reviews into English helps improve vocabulary consistency and may support more effective recurrent neural network learning.

A separate translated text column is created while preserving the original review text for reference, interpretability, and later error analysis.

In [ ]:
# Install / Import Translation Library

# This step requires internet access during the first translation run
try:

    # Import Google Translator
    from deep_translator import GoogleTranslator

    translator_available = True

except ImportError:

    try:
        # Install translation package if missing
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "deep-translator"]
        )

        from deep_translator import GoogleTranslator

        translator_available = True

    except Exception as e:

        print(
            "Translator could not be installed/imported:",
            e
        )

        translator_available = False





# Translation Function
# Function to translate Malay reviews into English
def translate_malay_to_english(text):

    # Return original text if translator unavailable
    if not translator_available:
        return text

    try:
        # Create translator object
        translator = GoogleTranslator(
            source="ms",
            target="en"
        )

        # Translate review text
        return translator.translate(str(text))

    except Exception as e:

        print(
            "Translation failed for one review. Keeping original text."
        )

        # Return original text if translation fails
        return text





# Translation Cache File
# Cache translated dataset to avoid repeated API translation
translation_cache_file = (
    "pizza_reviews_translated_partB.csv"
)





# Load Cached Translation If Available
if os.path.exists(translation_cache_file):

    # Load previously translated dataset
    df = pd.read_csv(
        translation_cache_file
    )

    print(
        "Loaded cached translated dataset."
    )

else:

    # Create translated review column
    df["Review_Translated"] = df["Review"]

    # Identify Malay reviews
    malay_mask = (
        df["Language"] == "Malay"
    )

    # Translate Malay reviews into English
    df.loc[
        malay_mask,
        "Review_Translated"
    ] = df.loc[
        malay_mask,
        "Review"
    ].apply(
        translate_malay_to_english
    )

    # Save translated dataset for reuse
    df.to_csv(
        translation_cache_file,
        index=False
    )

    print(
        "Translation completed and saved to cache."
    )





# Display Sample Translation Results
# Display sample translated reviews
df[
    [
        "Review",
        "Language",
        "Review_Translated"
    ]
].sample(
    10,
    random_state=42
)

## Insights

Malay reviews were successfully translated into English and stored in a separate `Review_Translated` column while preserving the original review text for reference and later error analysis.

English reviews remain unchanged, whereas Malay reviews are translated into English before text cleaning and tokenization. This helps create a more unified vocabulary space for downstream recurrent neural network modelling.

Without translation, tokenization would treat semantically similar English and Malay words as entirely separate tokens, increasing vocabulary fragmentation and reducing representation consistency. By translating the reviews into a common language, the preprocessing pipeline reduces multilingual vocabulary complexity and may improve the model’s ability to learn sentiment-related textual patterns more effectively.

Preserving the original review text is also important because it improves interpretability and allows future comparison between original and translated reviews during model evaluation or error analysis.

A translation cache file was created to avoid repeated API translation calls, improving notebook efficiency, reproducibility, and execution speed during subsequent runs.

Although translation may introduce minor semantic inaccuracies, the overall approach provides a practical balance between multilingual dataset retention and vocabulary consistency for deep learning-based sentiment classification.

## Step 5.5: Duplicate Review Consistency Cleaning

Some reviews may appear multiple times with different numerical scores. This creates inconsistent labels because identical review text may correspond to different sentiments. Such inconsistencies introduce noisy supervisory signals during model training and may negatively affect sentiment classification performance.

To improve label consistency, duplicate translated reviews are identified and cleaned before recurrent neural network modelling.

In [ ]:
# Count Unique Scores for Each Review


# Group reviews by translated review text and count
# how many different score values exist for each review.


duplicate_reviews = df.groupby(
    "Review_Translated"
)["Score"].nunique()



# Identify Reviews with Conflicting Scores

# Reviews with more than one unique score are considered
# conflicting duplicate reviews.


conflicting_reviews = duplicate_reviews[
    duplicate_reviews > 1
].index



# Remove Duplicate Reviews

# Keep only the first occurrence of duplicate translated
# reviews to improve label consistency.


df = df.drop_duplicates(
    subset=["Review_Translated"],
    keep="first"
)




# Display Cleaning Results
print(
    "Number of conflicting duplicate reviews:",
    len(conflicting_reviews)
)

print(
    "Dataset shape after duplicate cleaning:",
    df.shape
)

## Insights

The dataset contains duplicate reviews where identical translated review text may correspond to different numerical scores. This creates label inconsistency because the same textual input may be associated with multiple sentiment categories.

A total of 51 translated reviews were identified as having conflicting score values. Such contradictions can introduce noisy training signals and reduce the model’s ability to learn reliable sentiment patterns.

To improve dataset consistency, duplicate translated reviews were removed while retaining only the first occurrence of each review.

This preprocessing step helps reduce contradictory supervision during training and improves the reliability of downstream sentiment classification modelling. Cleaner label consistency is especially important in deep learning because neural networks are highly sensitive to noisy or conflicting target labels.

After duplicate cleaning, the dataset size was reduced from 893 reviews to 777 reviews, indicating that duplicate review occurrences were relatively common within the dataset.

## Step 6: Basic Text Cleaning

Basic text preprocessing is applied to the translated review text to standardise the textual input before tokenization and recurrent neural network modelling.

The cleaning process includes:

- Converting text to lowercase
- Removing punctuation and special characters
- Normalising whitespace

These preprocessing steps help reduce unnecessary textual variation and improve vocabulary consistency across reviews.

In [ ]:

# Text Cleaning Function

# Function to clean review text
def clean_text(text):

    # Convert text to lowercase
    text = str(text).lower()

    # Remove punctuation and special characters
    text = re.sub(
        r"[^\w\s]",
        "",
        text
    )

    # Remove extra whitespace
    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    return text




# Apply Text Cleaning
# Create cleaned review column
df["Clean_Review"] = df[
    "Review_Translated"
].apply(
    clean_text
)

# Display sample cleaned reviews
df[
    [
        "Review",
        "Review_Translated",
        "Clean_Review"
    ]
].head()

## Insights

The text cleaning process successfully standardised the translated review text into a cleaner format suitable for NLP preprocessing and recurrent neural network modelling.

The preprocessing pipeline converted all text into lowercase format, removed punctuation and special characters, and normalised whitespace. These steps help reduce unnecessary vocabulary variation and improve token consistency during tokenization.

For example, words such as:

- `"Pizza"`
- `"pizza!"`
- `"PIZZA"`

would all be transformed into the same standardised token representation (`pizza`).

Reducing vocabulary inconsistency is important in NLP because deep learning models treat differently formatted words as separate tokens. Without standardisation, semantically identical words could unnecessarily increase vocabulary size and reduce representation efficiency.

The cleaned review text stored in the `Clean_Review` column will serve as the primary textual input for subsequent tokenization, sequence generation, and sentiment classification modelling.

In [ ]:
# Display random examples of:
# - original review text
# - translated review text
# - cleaned review text

df[
    [
        "Review",
        "Review_Translated",
        "Clean_Review"
    ]
].sample(
    5,
    random_state=42
)

## Insights

The sample preprocessing results confirm that the translation and text cleaning pipeline is functioning as intended.

Malay reviews were successfully translated into English before cleaning, while English reviews remained unchanged. The cleaning process then standardised the text by converting it into lowercase format and removing punctuation and special characters.

The side-by-side comparison demonstrates the complete preprocessing workflow:

- Original review text
- Translated English representation
- Final cleaned NLP-ready text

This verification step is important because preprocessing errors may unintentionally distort semantic meaning or remove important textual information. Reviewing sample outputs helps ensure that the transformed reviews remain understandable while achieving greater vocabulary consistency.

Overall, the preprocessing pipeline successfully converts multilingual raw review text into a cleaner and more standardised format suitable for tokenization and recurrent neural network modelling.

# Step 7. Exploratory Data Analysis (EDA)

Exploratory Data Analysis (EDA) is performed to better understand the cleaned dataset before recurrent neural network modelling.

The analysis focuses on:

- sentiment distribution
- language distribution
- score patterns
- review length characteristics

EDA helps identify potential class imbalance, dataset characteristics, and preprocessing considerations that may affect sentiment classification performance.

## 7.1 Sentiment Distribution

In [ ]:
# Create figure size
plt.figure(figsize=(6,4))

# Plot sentiment class distribution
ax = sns.countplot(
    x="Sentiment",
    data=df,
    order=[
        "Negative",
        "Neutral",
        "Positive"
    ]
)

# Add chart title and axis labels
plt.title("Sentiment Distribution")
plt.xlabel("Sentiment")
plt.ylabel("Count")

# Display count labels on top of bars
for p in ax.patches:

    ax.annotate(
        f"{int(p.get_height())}",

        # Position labels above bars
        (
            p.get_x() + p.get_width() / 2.,
            p.get_height()
        ),

        ha="center",
        va="bottom"
    )

# Display plot
plt.show()

## Insights

The sentiment distribution confirms that the dataset contains all three sentiment categories with relatively balanced class representation after preprocessing and duplicate review cleaning.

Positive reviews form the largest class with 337 samples, followed by 221 Negative reviews and 219 Neutral reviews. Although the distribution is not perfectly equal, no single class overwhelmingly dominates the dataset.

This relatively balanced class distribution is beneficial for multiclass sentiment classification because it reduces the likelihood of recurrent neural network models becoming heavily biased toward a single sentiment category during training.

The visualisation also confirms that duplicate review cleaning did not severely distort the overall sentiment distribution, allowing the dataset to remain suitable for supervised learning and fairer evaluation across all sentiment categories.

Overall, the dataset structure remains appropriate for recurrent neural network-based sentiment classification and comparative model evaluation.

## 7.2 Language Distribution

The language distribution of the cleaned dataset is visualised to understand the relative representation of English and Malay reviews after filtering smaller language categories.

This analysis helps justify the multilingual preprocessing and translation decisions used before recurrent neural network modelling.

In [ ]:
# Create figure size
plt.figure(figsize=(6,4))

# Plot language distribution
ax = sns.countplot(
    x="Language",
    data=df
)

# Add chart title and axis labels
plt.title("Language Distribution")
plt.xlabel("Language")
plt.ylabel("Count")

# Display count labels above bars
for p in ax.patches:

    ax.annotate(
        f"{int(p.get_height())}",

        # Position labels above bars
        (
            p.get_x() + p.get_width() / 2.,
            p.get_height()
        ),

        ha="center",
        va="bottom"
    )

# Display plot
plt.show()

## Insights

The language distribution confirms that the cleaned dataset consists primarily of English and Malay reviews, with English reviews slightly outnumbering Malay reviews after preprocessing and duplicate cleaning.

The relatively balanced representation between the two language groups is beneficial because both languages contribute substantial amounts of training data for sentiment analysis modelling.

Importantly, the visualisation further justifies retaining and translating Malay reviews rather than discarding them entirely. Removing Malay reviews would significantly reduce the dataset size and potentially limit the diversity of sentiment expressions available for model learning.

The analysis also supports the decision to apply Malay-to-English translation before tokenization. By preserving both language groups while unifying them into a common English vocabulary space, the preprocessing pipeline can reduce multilingual vocabulary fragmentation while still benefiting from a larger and more diverse dataset.

Overall, the language distribution supports the multilingual preprocessing strategy used in this project and confirms that the dataset remains sufficiently representative for recurrent neural network-based sentiment classification.

## 7.3 Sentiment Distribution by Language

The relationship between language and sentiment distribution is analysed to determine whether both English and Malay reviews contribute meaningful sentiment diversity to the dataset.

This analysis helps evaluate whether retaining and translating Malay reviews is beneficial for multiclass sentiment classification.

In [ ]:
# Create figure size
plt.figure(figsize=(8, 5))

# Plot sentiment distribution grouped by language
ax = sns.countplot(
    x="Language",
    hue="Sentiment",
    data=df,

    # Control sentiment order in legend
    hue_order=[
        "Negative",
        "Neutral",
        "Positive"
    ]
)

# Add chart title and axis labels
plt.title("Sentiment Distribution by Language")
plt.xlabel("Language")
plt.ylabel("Count")

# Move legend outside graph area
plt.legend(
    title="Sentiment",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

# Improve layout spacing
plt.tight_layout()

# Display plot
plt.show()

## Insights

The sentiment distribution by language confirms that both English and Malay reviews contain meaningful representations of all three sentiment categories: Negative, Neutral, and Positive.

This indicates that Malay reviews are not sparse or sentiment-restricted data. Instead, they contribute useful linguistic and sentiment diversity that can support more effective sentiment classification modelling.

The visualisation further justifies retaining and translating Malay reviews rather than discarding them. Removing Malay reviews would substantially reduce dataset size and eliminate valuable sentiment-related textual patterns from the training data.

Additionally, Positive reviews remain the dominant sentiment category across both language groups, although the overall sentiment distribution still remains reasonably balanced for multiclass classification.

From an NLP perspective, preserving multilingual sentiment diversity while translating reviews into a common English vocabulary space may help the recurrent neural network learn richer sentiment representations without excessively increasing vocabulary fragmentation.

Overall, the analysis supports the multilingual preprocessing strategy used throughout the project and confirms that both language groups contribute meaningful information for sentiment classification.

### 7.4 Score Distribution

The distribution of numerical review scores is visualised to better understand the rating behaviour of the dataset before sentiment classification modelling.

This analysis helps evaluate whether the generated sentiment categories align reasonably with the underlying score patterns and whether the dataset contains sufficient variation across low, moderate, and high review scores.

In [ ]:
# Create figure size
plt.figure(figsize=(6,4))

# Plot numerical score distribution
sns.histplot(

    # Review score column
    df["Score"],

    # Number of histogram bins
    bins=20,

    # Display smooth density curve
    kde=True
)

# Add chart title and axis labels
plt.title("Score Distribution")
plt.xlabel("Score")
plt.ylabel("Count")

# Display plot
plt.show()

## Insights

The score distribution shows that the dataset contains a wide range of review ratings, spanning from very low scores to very high scores.

The histogram reveals noticeable clustering near the lower and upper ends of the score range, indicating that many reviews express strongly negative or strongly positive opinions. In contrast, moderate middle-range scores appear less frequently.

This suggests that the dataset exhibits a degree of sentiment polarization, which is commonly observed in real-world review datasets where users are more likely to leave reviews when they feel strongly satisfied or strongly dissatisfied.

The presence of both low-score and high-score clusters also indicates that the dataset contains meaningful sentiment variation, which is beneficial for supervised sentiment classification tasks.

Additionally, the observed score distribution supports the earlier sentiment label engineering strategy, where low scores were mapped to Negative sentiment, middle-range scores to Neutral sentiment, and high scores to Positive sentiment. The score patterns therefore align reasonably well with the generated multiclass sentiment categories.

#### Descriptive Statistics of Review Scores

Descriptive statistics are analysed to summarise the central tendency, variability, and distribution characteristics of the numerical review scores.

This analysis complements the earlier histogram visualisation by providing quantitative insights into the spread and overall behaviour of the rating distribution.

In [ ]:
# Display descriptive statistics for review scores
df["Score"].describe()

## Insights

The descriptive statistics confirm that the review scores span a wide numerical range, with scores varying from very low ratings to highly positive ratings.

The average review score is approximately 5.87, while the median score is approximately 5.45. Since the mean and median are relatively close, the dataset does not appear to be extremely skewed toward only low or high ratings overall.

The relatively large standard deviation (approximately 3.71) indicates substantial variation in customer opinions, which is beneficial for sentiment classification because the models are exposed to diverse sentiment patterns and rating behaviours.

The quartile values further support the earlier histogram analysis, showing that many reviews cluster nearer to the lower and upper score ranges rather than around moderate middle-range values. This suggests a degree of sentiment polarization within the dataset.

Overall, the descriptive statistics reinforce the earlier exploratory analysis and confirm that the dataset contains sufficient rating diversity for multiclass sentiment classification modelling.

In [ ]:
# Inspect reviews with scores below 0
df[df["Score"] < 0][
    [
        "Review",
        "Score",
        "Language",
        "Sentiment",
        "Review_Translated",
        "Clean_Review"
    ]
]

## Insights

The descriptive statistics show that review scores range from -0.5 to 10.0, with an average score of approximately 5.87 and a median score of approximately 5.45.

Some reviews contain scores slightly below 0. These records were inspected separately and found to contain clearly negative review text, such as comments about stale toppings, poor taste, feeling sick, or not wanting to order again.

Although negative numerical scores are unusual for a review rating scale, the review text confirms that these records still represent valid negative sentiment. Since they align with the sentiment labelling rule where scores below 4 are classified as Negative, they are retained rather than removed.

This inspection helps avoid removing valid negative examples unnecessarily. Retaining them preserves useful sentiment information and ensures that the model is exposed to strong negative review patterns during training.

Overall, the wide score range and large standard deviation indicate strong variation in customer opinions, which is useful for sentiment classification because the model can learn from diverse negative, neutral, and positive review patterns.

### 7.5 Boxplot of Review Scores

A boxplot is used to visualise the spread, central tendency, variability, and potential outliers within the numerical review score distribution.

This provides additional statistical insight into the overall score behaviour and helps identify whether extreme rating values may significantly affect the dataset before sentiment classification modelling.

In [ ]:
# Create figure size
plt.figure(figsize=(7,2.5))

# Plot boxplot of numerical review scores
sns.boxplot(
    x=df["Score"]
)

# Add chart title and axis label
plt.title("Boxplot of Review Scores")
plt.xlabel("Score")

# Display plot
plt.show()

## Insights

The boxplot confirms that the review scores are distributed across a broad numerical range, indicating substantial variation in customer opinions and sentiment intensity.

The median score appears near the middle of the score range, while the wide interquartile range suggests that many reviews differ considerably in their rating behaviour.

The visualisation also confirms the presence of both highly negative and highly positive reviews, supporting the earlier histogram analysis that the dataset exhibits meaningful sentiment diversity and some degree of sentiment polarization.

Although a few slightly negative score values exist, the boxplot does not reveal severe numerical outliers that would heavily distort the overall score distribution. This further supports the earlier decision to retain the negative-valued reviews because they appear consistent with the overall dataset structure rather than being isolated anomalies.

Overall, the score distribution remains sufficiently stable and diverse for multiclass sentiment classification modelling.

### 7.6 Review Length Analysis

The distribution of review lengths is analysed to better understand the textual sequence characteristics of the dataset before tokenization and recurrent neural network modelling.

Review length analysis is important because sequence length directly affects tokenization, sequence padding, truncation behaviour, and the ability of recurrent neural networks to capture contextual information from textual sequences.

In [ ]:

# Calculate Review Length

# Count number of words in each cleaned review
df["Review_Length"] = df[
    "Clean_Review"
].apply(
    lambda x: len(x.split())
)




# Plot Review Length Distribution


# Create figure size
plt.figure(figsize=(6,4))

# Plot histogram of review lengths
sns.histplot(

    # Review length column
    df["Review_Length"],

    # Number of histogram bins
    bins=30,

    # Display smooth density curve
    kde=True
)

# Add chart title and axis labels
plt.title("Review Length Distribution")
plt.xlabel("Number of Words")
plt.ylabel("Count")

# Display plot
plt.show()

## Insights

The review length distribution shows that most reviews are relatively short, with the majority containing fewer than 15 words.

The distribution is right-skewed, meaning that short reviews are much more common than very long reviews. This behaviour is typical in real-world customer review datasets where users often provide brief opinions, reactions, or concise feedback.

The analysis also shows that only a small number of reviews contain very long textual sequences. This observation supports the later decision to apply sequence padding with a maximum sequence length of 100, since this length is sufficient to capture nearly all review content without excessive truncation.

From an NLP perspective, shorter reviews may contain limited contextual information, which can make sentiment classification more challenging for certain samples, especially Neutral or ambiguous opinions. In contrast, longer reviews may provide richer contextual patterns that recurrent neural networks can use to better learn sentiment relationships.

Overall, the review length distribution confirms that the dataset contains manageable sequence lengths suitable for recurrent neural network-based sentiment analysis.

#### Descriptive Statistics of Review Length

Descriptive statistics are analysed to quantitatively summarise the distribution, variability, and overall characteristics of review lengths within the dataset.

This analysis complements the earlier visualisation and helps guide sequence padding and tokenization decisions for recurrent neural network modelling.

In [ ]:
# Display descriptive statistics for review lengths
df["Review_Length"].describe()

## Insights

The descriptive statistics confirm that most reviews are relatively short, with an average review length of approximately 9.11 words and a median length of 8 words.

The quartile values show that 75% of reviews contain 11 words or fewer, indicating that the dataset is dominated by short textual sequences rather than long paragraphs or detailed reviews.

The maximum review length is 49 words, which strongly supports the later decision to use a maximum sequence length of 100 during sequence padding. This value is sufficiently large to capture nearly all review content without significant truncation.

The relatively short review lengths also suggest that the sentiment classification task may be more challenging for certain samples because shorter reviews often contain limited contextual information and more ambiguous wording.

From an NLP perspective, short-text sentiment classification can be difficult because models must infer sentiment from fewer contextual cues. However, the compact sequence lengths also reduce computational complexity and make the dataset well-suited for recurrent neural network training.

### 7.7 Review Length by Sentiment

The relationship between review length and sentiment category is analysed to investigate whether certain sentiment classes tend to contain longer or shorter textual sequences.

This analysis provides additional insight into how sequence length, contextual information, and textual complexity may influence sentiment classification difficulty in recurrent neural network models.

In [ ]:
# Create figure size
plt.figure(figsize=(7,4))

# Plot review length distribution by sentiment class
sns.boxplot(

    # Sentiment categories on x-axis
    x="Sentiment",

    # Review length values on y-axis
    y="Review_Length",

    # Dataset
    data=df,

    # Control sentiment order
    order=[
        "Negative",
        "Neutral",
        "Positive"
    ]
)

# Add chart title and axis labels
plt.title("Review Length by Sentiment")
plt.xlabel("Sentiment")
plt.ylabel("Number of Words")

# Display plot
plt.show()

## Insights

The boxplot shows that review length varies across all three sentiment categories, although most reviews remain relatively short overall.

Positive reviews appear to have slightly longer median review lengths compared to Negative reviews, while Neutral reviews show a wider spread of sequence lengths. However, there is still substantial overlap between all sentiment categories.

This overlap indicates that review length alone is not sufficient to reliably determine sentiment polarity. Instead, recurrent neural network models must learn meaningful semantic and contextual patterns from the textual content itself rather than relying purely on sequence length.

The visualisation also suggests that shorter reviews may contribute to greater sentiment ambiguity because they often contain limited contextual information or vague wording. This may increase classification difficulty, particularly for Neutral sentiment reviews where opinions are less strongly expressed.

Overall, the analysis confirms that the dataset contains diverse textual sequence lengths while still remaining dominated by concise customer reviews suitable for recurrent neural network modelling.

### 7.8 Score Outlier Analysis

Boxplots and interquartile range (IQR) analysis are used to investigate whether unusually extreme review scores exist within the dataset.

Outlier analysis helps determine whether extreme numerical scores should be retained or removed before sentiment classification modelling.

In [ ]:

# Display Boxplot of Review Scores

# Visualise the score distribution and identify
# possible outliers.


plt.figure(figsize=(8,4))

sns.boxplot(
    x=df["Score"]
)

plt.title("Pizza Review Score Distribution")

plt.xlabel("Score")

plt.show()



# Compute Quartiles and IQR

#
# Q1 = 25th percentile
# Q3 = 75th percentile
# IQR = Interquartile Range

Q1 = df["Score"].quantile(0.25)

Q3 = df["Score"].quantile(0.75)

IQR = Q3 - Q1



# Compute Outlier Boundaries

# Scores outside these boundaries are considered
# potential outliers.
lower_bound = Q1 - 1.5 * IQR

upper_bound = Q3 + 1.5 * IQR



# Identify Outlier Reviews
outliers = df[
    (df["Score"] < lower_bound)
    |
    (df["Score"] > upper_bound)
]



# Display Outlier Results
print(
    "Number of score outliers:",
    len(outliers)
)

print("\nSample Outlier Reviews:")

print(
    outliers[
        [
            "Review_Translated",
            "Score"
        ]
    ].head()
)

## Insights

Score outlier analysis was performed using boxplots and interquartile range (IQR) analysis to investigate the presence of unusually extreme review scores.

The analysis detected no numerical score outliers based on the IQR outlier boundaries. This suggests that the review scores remain relatively well distributed within the expected scoring range despite the presence of slightly negative values.

The result further supports the earlier decision to retain the negative review scores because they do not behave as statistically isolated anomalies within the dataset.

From a sentiment analysis perspective, retaining the full range of valid review scores helps preserve meaningful sentiment diversity and allows the recurrent neural network models to learn from both strongly negative and strongly positive review patterns.

Overall, the score distribution appears stable and suitable for downstream multiclass sentiment classification modelling.

# Step 8: Problem Formulation and Decision-Making

## 8.1 Classification vs Regression Decision

The original `Score` column is numerical, meaning that both regression and classification approaches are possible.

This project uses classification because the primary objective is to analyse customer sentiment rather than predict an exact numerical score value. The numerical scores are therefore converted into three interpretable sentiment categories:

- Negative
- Neutral
- Positive

Classification provides clearer business and customer insight because sentiment categories are easier to interpret than continuous numerical predictions. It also allows model performance to be evaluated using classification-specific metrics such as:

- Accuracy
- Precision
- Recall
- F1-score
- Confusion matrices

Additionally, sentiment classification aligns more closely with real-world customer feedback analysis tasks, where understanding opinion polarity is often more valuable than predicting an exact numerical rating.

---

## 8.2 Language Handling Decision

English and Malay reviews are retained because they form the overwhelming majority of the dataset and both contribute meaningful sentiment diversity.

Instead of removing Malay reviews, the Malay review text is translated into English before tokenization and modelling. This helps reduce vocabulary fragmentation because semantically similar English and Malay words can be represented within a more unified English vocabulary space.

Without translation, tokenization would treat English and Malay words as separate vocabulary tokens even when they express highly similar meanings. Translating the reviews therefore improves vocabulary consistency and may support more effective sequence learning during recurrent neural network training.

Very small language categories are removed because they contain insufficient samples to support reliable training, meaningful evaluation, or robust model generalisation.

---

## 8.3 RNN Architecture Decision

Recurrent Neural Network (RNN)-based architectures are selected because text data is sequential in nature, where word order contributes significantly to semantic meaning and sentiment interpretation.

A simple baseline RNN architecture is first developed to establish initial sentiment classification performance. More advanced recurrent architectures, including LSTM and GRU models, are then implemented and compared to improve contextual learning capability and overall classification performance.

LSTM and GRU architectures are particularly suitable for NLP tasks because they are specifically designed to better capture sequential dependencies and contextual relationships across word sequences compared to traditional RNN models. These architectures also help mitigate issues such as short-term memory limitations and vanishing gradients commonly encountered in standard RNN networks.

# Step 9: Feature Engineering and Data Augmentation Planning

## 9.1 Review Length Feature

The previously created `Review_Length` feature is inspected to verify that each cleaned review has been assigned an appropriate word count.

Although the recurrent neural network models primarily use tokenized text sequences for prediction, review length remains useful for exploratory analysis, sequence-length justification, preprocessing decisions, and later error analysis.

In [ ]:
# Display sample reviews with their cleaned text and review length
df[
    [
        "Review",
        "Clean_Review",
        "Review_Length"
    ]
].sample(
    5,
    random_state=42
)

## Insights

The sample output confirms that the `Review_Length` feature was correctly generated from the cleaned review text.

The examples show that Malay reviews were translated into English before cleaning and word-count calculation, ensuring that review length is measured consistently using the final text representation applied during modelling.

Although `Review_Length` is not directly used as an explicit input feature for the recurrent neural network models, it remains valuable for understanding sequence-length behaviour, justifying padding decisions, and supporting later error analysis.

This feature also helps identify whether extremely short reviews may contribute to greater sentiment ambiguity due to limited contextual information. In contrast, longer reviews may provide richer semantic patterns that can support more effective sentiment learning.

Overall, the review length feature provides useful supporting information for understanding dataset characteristics and sequence modelling considerations.

## 9.2 Sentence Splitting as Possible Data Augmentation

Some reviews contain multiple sentences expressing different opinions or contextual details.

To investigate whether additional training data could be generated, reviews containing multiple sentences are identified as potential candidates for sentence-level augmentation.

This analysis helps evaluate whether splitting longer reviews into shorter sentence-based samples may increase dataset size and improve recurrent neural network learning performance.

In [ ]:

# Identify Multi-Sentence Reviews

# Select reviews containing multiple sentences
multi_sentence_reviews = df[
    df["Review_Translated"].str.contains(

        # Detect multiple sentence punctuation marks
        r"[.!?].*[.!?]",

        regex=True,
        na=False
    )
]

# Display sample multi-sentence reviews
multi_sentence_reviews[
    [
        "Review_Translated",
        "Score",
        "Sentiment"
    ]
].head()

## Insights

The sample output confirms that many reviews contain multiple sentences, making sentence-level splitting technically possible as a form of data augmentation.

Several reviews contain richer contextual information across multiple sentences, including combinations of positive and negative opinions within the same review. This suggests that sentence splitting may increase the number of training samples available for recurrent neural network modelling.

However, the examples also reveal an important limitation of sentence-level augmentation. Some reviews contain mixed sentiment expressions, meaning that individual sentences may not fully align with the overall review sentiment label inherited from the original review.

For example, a review labelled as Neutral may contain one positive sentence and one negative sentence. Splitting such reviews into isolated sentence-level samples could therefore introduce label inconsistency and noisy supervision during model training.

Additionally, sentence splitting may remove important contextual relationships between sentences, potentially reducing semantic coherence within the training data.

These observations justify why sentence-level augmentation is evaluated experimentally later in the project rather than being automatically assumed to improve model performance.

In [ ]:
# Display number of multi-sentence reviews
print(
    "Number of multi-sentence reviews:",
    len(multi_sentence_reviews)
)

# Display percentage of multi-sentence reviews
print(
    "Percentage of multi-sentence reviews:",
    round(
        len(multi_sentence_reviews) / len(df) * 100,
        2
    ),
    "%"
)

## Insights

The analysis shows that 155 reviews, representing approximately 19.95% of the dataset, contain multiple sentences and are therefore potential candidates for sentence-level augmentation.

Although most reviews remain relatively short single-sentence comments, the number of multi-sentence reviews is still sufficiently large to justify investigating sentence splitting as a possible method for increasing training data size.

The results also suggest that sentence-level augmentation would likely produce only a moderate dataset expansion rather than a dramatic increase in training samples.

Additionally, the relatively low proportion of multi-sentence reviews is consistent with the earlier review length analysis, which showed that the dataset is dominated by concise customer reviews.

Overall, the analysis indicates that sentence-level augmentation may provide some additional training diversity, but its effectiveness should be evaluated carefully due to the potential risk of label inconsistency and contextual fragmentation discussed earlier.

# Step 10: Text Preprocessing

In this revised workflow, tokenization is performed before the train-test split to construct a unified vocabulary representation using the complete cleaned text dataset before separating the data into training and testing sets.

Building the tokenizer on the full cleaned dataset helps ensure that the vocabulary representation remains consistent across both training and testing data.

---

## 10.1 Encode Sentiment Labels

The sentiment categories are converted into numerical labels because recurrent neural network models require numerical target values during training.

In [ ]:

# Encode Sentiment Labels

# Create label encoder object
encoder = LabelEncoder()

# Convert sentiment categories into numerical labels
y = encoder.fit_transform(
    df["Sentiment"]
)

# Display first 10 encoded labels
print(y[:10])

# Display class mapping
print("\nClasses:")
print(encoder.classes_)

## Insights

The sentiment categories were successfully converted into numerical labels using label encoding.

The resulting mapping is:

- Negative → 0
- Neutral → 1
- Positive → 2

This preprocessing step is necessary because recurrent neural network models require numerical target representations during supervised training and evaluation.

Encoding the sentiment labels also enables the models to compute classification loss functions, generate predictions, and evaluate performance using classification-specific metrics such as accuracy, precision, recall, F1-score, and confusion matrices.

The encoded labels will therefore serve as the target variables throughout the sentiment classification experiments.

In [ ]:

# Display Label Mapping

# Create dictionary showing sentiment-to-label mapping
label_mapping = dict(

    zip(

        # Original sentiment labels
        encoder.classes_,

        # Encoded numerical labels
        encoder.transform(
            encoder.classes_
        )
    )
)

# Display label mapping
print("Label Mapping:")
print(label_mapping)

## Insights

The label mapping confirms that each sentiment category was successfully converted into a consistent numerical representation for recurrent neural network training.

The mapping used throughout the project is:

- Negative → 0
- Neutral → 1
- Positive → 2

Maintaining a consistent label mapping is important because the encoded numerical values will be used throughout the entire modelling pipeline, including model training, prediction generation, classification report evaluation, and confusion matrix interpretation.

Clear documentation of the label mapping also improves interpretability and helps ensure that predicted numerical outputs can be correctly translated back into their corresponding sentiment categories during analysis.

## 10.2 Lemmatization Preprocessing

Lemmatization reduces words into their root forms while preserving semantic meaning.

Examples:

- `"running"` → `"run"`
- `"likes"` → `"like"`

This preprocessing step helps reduce vocabulary fragmentation and improves word consistency during tokenization and embedding generation.

In [ ]:

# Initialize WordNet Lemmatizer

lemmatizer = WordNetLemmatizer()



# Define Lemmatization Function

def lemmatize_text(text):

    # Split sentence into individual words
    words = word_tokenize(text)

    # Lemmatize each word
    lemmatized_words = [
        lemmatizer.lemmatize(word, pos="v")
        for word in words
    ]

    # Join words back into sentence
    return " ".join(lemmatized_words)



# Apply Lemmatization to Clean Reviews

df["Clean_Review"] = df[
    "Clean_Review"
].apply(lemmatize_text)



# Display Sample Results
print(
    df[
        [
            "Review_Translated",
            "Clean_Review"
        ]
    ].head()
)

## Insights

Lemmatization was applied to reduce words into their root forms while preserving semantic meaning.

This preprocessing step helps reduce vocabulary fragmentation by ensuring that different grammatical variations of similar words are represented more consistently during tokenization and embedding generation.

For example, words such as `"likes"`, `"liked"`, and `"liking"` can be normalised into a more consistent root representation. This helps reduce unnecessary vocabulary expansion and improves the efficiency of sequence representation learning.

Lemmatization may also improve the model’s ability to learn meaningful sentiment patterns across reviews containing different grammatical word forms while still preserving important semantic context.

## 10.3 Fit Tokenizer on Full Cleaned Text Dataset

The tokenizer is fitted on the complete cleaned review dataset before the train-test split.

This workflow allows the tokenizer to construct a unified vocabulary representation using all available cleaned text data before sequence generation and padding.

For this educational sentiment analysis project, fitting the tokenizer on the full dataset improves vocabulary coverage and reduces the likelihood of unseen words appearing during testing.

In [ ]:

# Tokenizer Configuration

# Maximum vocabulary size
max_words = 5000

# Maximum sequence length for padding
max_length = 100


# Create Tokenizer


tokenizer = Tokenizer(

    # Maximum number of vocabulary words
    num_words=max_words,

    # Token for unknown words
    oov_token="<OOV>"
)



# Fit Tokenizer on Full Cleaned Dataset
# Build tokenizer vocabulary using cleaned reviews
tokenizer.fit_on_texts(
    df["Clean_Review"]
)

print("Tokenizer fitted successfully.")

## Insights

The tokenizer was successfully fitted on the complete cleaned review dataset before the train-test split.

This approach allows the tokenizer to construct a more comprehensive vocabulary representation using all available cleaned text data, which is beneficial for this relatively small educational sentiment analysis dataset.

The tokenizer vocabulary will later be used to convert cleaned review text into numerical token sequences suitable for recurrent neural network training.

Additionally, the use of an `<OOV>` token helps manage unseen or rare words by mapping them into a consistent out-of-vocabulary representation rather than discarding them entirely. This improves model robustness when encountering unfamiliar vocabulary during testing or prediction.

The selected maximum sequence length of 100 is also supported by the earlier review length analysis, which showed that the longest review contains fewer than 50 words. Therefore, the chosen padding length is sufficiently large to capture nearly all review content without excessive truncation.

## 10.4 Convert Text to Sequences

The cleaned review text is converted into numerical token sequences using the fitted tokenizer vocabulary.

This transformation is necessary because recurrent neural network models cannot directly process raw textual data. Instead, each review must be represented as a sequence of numerical word indices before sequence padding and model training.

In [ ]:

# Convert Cleaned Text into Numerical Sequences

# Convert cleaned review text into token sequences
X_sequences = tokenizer.texts_to_sequences(
    df["Clean_Review"]
)

# Display first 5 tokenized sequences
print(X_sequences[:5])

## Insights

The cleaned review text was successfully converted into numerical token sequences using the fitted tokenizer vocabulary.

Each integer within a sequence represents a specific word index learned by the tokenizer during vocabulary construction. This transformation allows textual review data to be represented numerically so that recurrent neural network models can process sequential language patterns.

The generated sequences vary in length because different reviews contain different numbers of words and contextual information. Consequently, sequence padding is required in the next preprocessing step to ensure that all input sequences share a consistent fixed length before model training.

Converting text into numerical sequences is a fundamental NLP preprocessing step because deep learning models operate on numerical tensor representations rather than raw textual input.

## 10.5 Vocabulary Analysis

The tokenizer vocabulary is inspected to better understand the textual characteristics of the cleaned translated review dataset.

Vocabulary analysis helps verify that the tokenizer successfully captured meaningful pizza-related, sentiment-related, and domain-specific words before recurrent neural network modelling.

In [ ]:
# Vocabulary Analysis

# Access tokenizer vocabulary dictionary
word_index = tokenizer.word_index

# Display total vocabulary size
print(
    "Vocabulary Size:",
    len(word_index)
)

# Display first 10 indexed vocabulary words
top_words = list(
    word_index.items()
)[:10]

print("\nTop 10 Indexed Words:")

for word, index in top_words:

    print(f"{word}: {index}")

## Insights

The tokenizer successfully constructed a vocabulary from the cleaned translated review dataset, resulting in a total vocabulary size of 645 unique words.

The most frequent indexed words include common English terms as well as pizza-related and sentiment-related vocabulary such as `"pizza"`, `"sambal"`, `"fusion"`, and `"mayo"`. This indicates that the tokenizer effectively captured meaningful domain-specific textual patterns from the review corpus.

The presence of the `<OOV>` token also confirms that unseen or rare words can be handled consistently during sequence processing and model inference.

Additionally, the vocabulary now primarily consists of English-based tokens because Malay reviews were translated before tokenization. This helps reduce vocabulary fragmentation and creates a more consistent textual representation for recurrent neural network training.

The relatively compact vocabulary size is also beneficial for this dataset because it reduces unnecessary vocabulary sparsity while still preserving meaningful semantic diversity for sentiment classification.

## 10.6 Padding Sequences

The tokenized review sequences are padded to a fixed length so that all input samples share consistent dimensions before recurrent neural network training.

Padding is necessary because different reviews contain different numbers of words, while neural network models require fixed-size numerical input tensors for efficient batch processing and sequence learning.

In [ ]:

# Pad Token Sequences

# Pad all sequences to fixed length
X = pad_sequences(

    # Tokenized review sequences
    X_sequences,

    # Maximum sequence length
    maxlen=max_length
)

# Display padded dataset shape
print(
    "Padded input shape:",
    X.shape
)

# Display first padded sequence
print(X[0])

## Insights

All tokenized review sequences were successfully padded to a fixed length of 100 tokens, resulting in a final input shape of `(777, 100)`.

Padding is necessary because recurrent neural network models require input sequences with consistent dimensions for efficient batch training and tensor processing.

Shorter reviews were padded with zeros at the beginning of the sequence, while longer reviews would be truncated if they exceeded the maximum sequence length. However, the earlier review length analysis showed that the longest review contains fewer than 50 words, meaning that the selected maximum sequence length of 100 is sufficiently large to preserve nearly all review content without significant truncation.

The padded sequences now form the final numerical input representation used for recurrent neural network modelling.

# Step 11: Train-Test Split

After tokenization and sequence padding, the dataset is divided into training and testing sets using an 80:20 ratio.

The training set is used for recurrent neural network learning, while the testing set is reserved for final model evaluation on unseen data.

Stratified sampling is applied to preserve the sentiment class distribution across both datasets, helping ensure fairer and more reliable model evaluation.

In [ ]:

# Train-Test Split

# Split dataset into training and testing sets
X_train, X_test, y_train, y_test, train_indices, test_indices = train_test_split(

    # Input sequences
    X,

    # Encoded sentiment labels
    y,

    # Original dataframe indices
    df.index,

    # 80:20 split ratio
    test_size=0.2,

    # Reproducibility
    random_state=42,

    # Preserve class distribution
    stratify=y
)

# Display training and testing shapes
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

## Insights

The dataset was successfully divided into training and testing sets using an 80:20 split ratio.

The resulting padded input matrices contain:

- 621 training samples
- 156 testing samples

with each review represented as a fixed-length sequence of 100 tokens.

Stratified sampling was applied to preserve the sentiment class distribution across both datasets. This helps ensure that the training and testing sets remain representative of the original dataset and supports fairer and more reliable model evaluation.

The testing set will later serve as unseen data for evaluating the generalisation performance of the recurrent neural network models.

# Step 12: Baseline Simple RNN Model

A baseline Simple Recurrent Neural Network (SimpleRNN) model is first developed to establish an initial benchmark for sentiment classification performance before evaluating more advanced recurrent architectures such as LSTM and GRU.

The baseline model provides a reference point for comparing whether gated recurrent architectures improve contextual learning capability and overall sentiment classification performance.

In [ ]:

# Import SimpleRNN Layer

from tensorflow.keras.layers import SimpleRNN



# Build Baseline SimpleRNN Model
# Create sequential neural network model
model = Sequential(
    name="Baseline_RNN_Model"
)



# Embedding Layer
model.add(

    Embedding(

        # Vocabulary size
        max_words,

        # Embedding vector dimension
        32,

        # Input sequence length
        input_length=max_length
    )
)



# SimpleRNN Layer
model.add(

    SimpleRNN(

        # Number of recurrent units
        64
    )
)



# Output Layer
model.add(

    Dense(

        # Number of sentiment classes
        3,

        # Multi-class probability activation
        activation="softmax"
    )
)




# Compile Model
model.compile(

    # Multi-class classification loss function
    loss="sparse_categorical_crossentropy",

    # Optimizer
    optimizer="adam",

    # Evaluation metric
    metrics=["accuracy"]
)

# Display model architecture summary
model.summary()

## Insights

A baseline Simple Recurrent Neural Network (SimpleRNN) model was successfully developed as the initial recurrent architecture for sentiment classification.

The model consists of:

- An Embedding layer for converting tokenized words into dense vector representations
- A SimpleRNN layer with 64 recurrent units for sequential contextual learning
- A Dense output layer with softmax activation for multi-class sentiment prediction

The Embedding layer contains the majority of trainable parameters because each vocabulary word is represented using a learned 32-dimensional dense vector representation.

The SimpleRNN architecture serves as a foundational recurrent neural network baseline before evaluating more advanced gated recurrent architectures such as LSTM and GRU.

Although SimpleRNN models are computationally simpler and faster to train, they may struggle to capture longer contextual dependencies due to vanishing gradient limitations during sequence learning. This motivates the later evaluation of LSTM and GRU architectures, which are specifically designed to improve long-term sequential memory and contextual learning capability.

In [ ]:
# Display model input shape
print(model.input_shape)

# Display model output shape
print(model.output_shape)

## Insights

The model input shape confirms that each review is represented as a padded sequence containing 100 token positions.

The output shape of `(None, 3)` confirms that the model generates probability predictions across the three sentiment classes:

- Negative
- Neutral
- Positive

The softmax output layer enables the model to produce normalized probability distributions across all sentiment categories, making it suitable for multiclass sentiment classification.

These input and output dimensions are also fully consistent with the earlier preprocessing pipeline, including tokenization, sequence padding, and label encoding.

# Step 13: Train Baseline SimpleRNN Model

The baseline SimpleRNN model is trained using the padded token sequences and encoded sentiment labels.

Training and validation metrics are monitored across epochs to evaluate whether the model is learning meaningful sentiment patterns from the review dataset while maintaining reasonable generalisation performance.

In [ ]:
# Train Baseline SimpleRNN Model

history = model.fit(

    # Training input sequences
    X_train,

    # Training sentiment labels
    y_train,

    # Number of training epochs
    epochs=5,

    # Batch size for gradient updates
    batch_size=32,

    # Reserve 20% of training data for validation
    validation_split=0.2
)

## Insights

The baseline SimpleRNN model was successfully trained for 5 epochs using the padded review sequences and encoded sentiment labels.

Both training accuracy and validation accuracy improved steadily across epochs, while training and validation losses decreased consistently. This indicates that the model was able to learn meaningful sentiment-related patterns from the review dataset.

By the final epoch, the model achieved approximately 81.9% training accuracy and 78.4% validation accuracy. The relatively close relationship between training and validation performance suggests that the model is learning effectively while still maintaining reasonable generalisation capability.

Although the baseline SimpleRNN model demonstrates effective sequential learning performance, standard RNN architectures may still struggle to capture longer contextual dependencies due to vanishing gradient limitations. This motivates the later evaluation of more advanced gated recurrent architectures such as LSTM and GRU, which are specifically designed to improve long-term contextual learning.

# Step 14: Evaluate Baseline SimpleRNN Model

The baseline SimpleRNN model is evaluated using the unseen testing dataset to measure its generalisation performance on sentiment classification.

The resulting benchmark performance will later be compared against more advanced recurrent architectures such as LSTM and GRU.

In [ ]:
# Evaluate Baseline SimpleRNN Model

# Evaluate model performance on unseen testing data
loss, acc = model.evaluate(
    X_test,
    y_test
)

# Display testing loss
print(
    f"Baseline SimpleRNN Loss: {loss:.4f}"
)

# Display testing accuracy
print(
    f"Baseline SimpleRNN Accuracy: {acc:.4f}"
)

## Insights

The baseline SimpleRNN model achieved a test accuracy of approximately 76.92% with a test loss of approximately 0.6635 on the unseen testing dataset.

This result indicates that the SimpleRNN architecture was able to learn meaningful sentiment-related sequential patterns from the translated pizza review dataset.

The testing accuracy is also reasonably close to the earlier validation accuracy, suggesting that the model demonstrates acceptable generalisation performance without severe overfitting.

Although the baseline SimpleRNN model provides a useful initial benchmark, its simpler recurrent architecture may still struggle to capture longer contextual dependencies due to vanishing gradient limitations. This motivates the later evaluation of more advanced gated recurrent architectures such as LSTM and GRU, which are specifically designed to improve long-term sequential learning capability.

# Step 15: Baseline SimpleRNN Classification Report and Confusion Matrix

The baseline SimpleRNN model predictions are evaluated using a classification report and confusion matrix to analyse sentiment classification performance across all three sentiment classes.

These evaluation metrics provide deeper insight into precision, recall, F1-score, and class-specific prediction behaviour beyond overall accuracy alone.

In [ ]:
# Generate Predictions

# Predict class probabilities for testing dataset
y_pred = model.predict(X_test)

# Convert probability predictions into class labels
y_pred_classes = np.argmax(
    y_pred,
    axis=1
)



# Display Classification Report
print(

    classification_report(

        # True labels
        y_test,

        # Predicted labels
        y_pred_classes,

        # Sentiment class names
        target_names=encoder.classes_
    )
)

## Insights

The baseline SimpleRNN model achieved an overall test accuracy of approximately 77% on the unseen testing dataset.

The model performed strongest on Positive reviews, achieving an F1-score of 0.86 and a recall of 0.94. This suggests that strongly positive sentiment expressions were generally easier for the model to identify correctly.

Negative reviews achieved moderate performance, with both precision and recall around 0.70, indicating that the model was reasonably effective at identifying negative sentiment patterns.

However, the Neutral class produced the weakest performance, with a recall of 0.57 and an F1-score of 0.65. This suggests that neutral reviews are more difficult for the SimpleRNN model to distinguish, likely because neutral opinions often contain weaker, mixed, or more ambiguous sentiment expressions.

Overall, the baseline SimpleRNN model provides a useful initial benchmark, but the weaker Neutral classification performance suggests that more advanced recurrent architectures such as LSTM and GRU may be needed to better capture contextual sentiment patterns and improve class separation.

In [ ]:
# Generate Confusion Matrix

# Create confusion matrix
cm = confusion_matrix(
    y_test,
    y_pred_classes
)

# Create figure
plt.figure(figsize=(6,4))

# Plot confusion matrix heatmap
sns.heatmap(

    cm,

    # Display values inside boxes
    annot=True,

    # Integer formatting
    fmt="d",

    # Colour map
    cmap="Blues",

    # Predicted class labels
    xticklabels=encoder.classes_,

    # Actual class labels
    yticklabels=encoder.classes_
)

# Plot title
plt.title("Baseline SimpleRNN Confusion Matrix")

# Axis labels
plt.xlabel("Predicted Sentiment")
plt.ylabel("Actual Sentiment")

# Display plot
plt.show()

## Insights

The confusion matrix provides detailed class-level evaluation of the baseline SimpleRNN model beyond overall accuracy alone.

The model performs strongest on Positive reviews, correctly classifying 64 Positive samples. This suggests that strongly positive sentiment expressions were generally easier for the SimpleRNN architecture to learn and distinguish.

Negative reviews achieved moderate classification performance, with 31 correctly classified samples. However, some Negative reviews were misclassified as Neutral or Positive, indicating occasional difficulty in distinguishing weaker negative sentiment expressions.

Neutral reviews remain the most difficult class to classify correctly, with only 25 correctly classified samples. Many Neutral reviews were incorrectly predicted as either Negative or Positive, suggesting that neutral sentiment often contains more ambiguous, mixed, or less emotionally explicit wording.

The confusion matrix also shows that most classification errors occur between adjacent sentiment categories rather than between strongly opposite sentiments. This behaviour is common in sentiment analysis tasks where contextual polarity may not always be sharply separated.

Overall, the confusion matrix highlights the limitations of the baseline SimpleRNN architecture and provides motivation for evaluating more advanced recurrent architectures such as LSTM and GRU to improve contextual understanding and class separation.

# Step 16: LSTM Model

A more advanced LSTM architecture is developed to improve upon the baseline SimpleRNN model by increasing model capacity and introducing regularisation techniques.

The improved architecture aims to capture richer contextual sentiment patterns while reducing overfitting and improving generalisation performance.

In [ ]:
# Build LSTM Model

# Create sequential model
improved_model = Sequential(
    name="LSTM_Model"
)


# Embedding Layer
improved_model.add(

    Embedding(

        # Vocabulary size
        max_words,

        # Larger embedding dimension
        64,

        # Input sequence length
        input_length=max_length
    )
)


# LSTM Layer
improved_model.add(

    LSTM(

        # Number of LSTM units
        128,

        # Output final hidden state only
        return_sequences=False
    )
)



# Dropout Regularisation
improved_model.add(

    Dropout(0.5)
)




# Dense Hidden Layer
improved_model.add(

    Dense(

        64,
        activation="relu"
    )
)



# Output Layer
improved_model.add(

    Dense(

        3,
        activation="softmax"
    )
)



# Compile Model
improved_model.compile(

    loss="sparse_categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)

# Display architecture summary
improved_model.summary()

## Insights

The LSTM model introduces a more advanced recurrent architecture compared to the earlier baseline SimpleRNN model.

Several architectural improvements were implemented, including:

- a larger embedding dimension
- increased LSTM units
- dropout regularisation
- an additional Dense hidden layer

These modifications increase the model’s ability to learn richer semantic and contextual sentiment representations from the translated review dataset.

Unlike standard SimpleRNN architectures, LSTM networks use gated memory mechanisms that help preserve important sequential information across longer text dependencies while reducing vanishing gradient limitations during training.

The improved model also contains substantially more trainable parameters than the baseline SimpleRNN model, indicating increased learning capacity. However, the larger architecture also introduces greater overfitting risk, which is why dropout regularisation is incorporated to improve generalisation performance.

In [ ]:
# Display improved model input shape
print(improved_model.input_shape)

# Display improved model output shape
print(improved_model.output_shape)

## Insights

The LSTM model maintains the same input and output dimensions as the baseline SimpleRNN model.

Each review is represented as a padded sequence containing 100 token positions, while the output layer generates probability predictions across the three sentiment classes:

- Negative
- Neutral
- Positive

Maintaining consistent input and output dimensions helps ensure fairer experimental comparison between the baseline SimpleRNN model and the more advanced LSTM architecture because both models are trained and evaluated using the same preprocessed dataset structure.

This consistency allows performance differences to be attributed more directly to architectural improvements rather than changes in input representation or preprocessing configuration.

## 16.1 Configure LSTM Callbacks

Training callbacks are configured to improve model training stability, reduce overfitting risk, and preserve the best-performing model during training.

EarlyStopping is used to stop training automatically when validation performance stops improving, while ModelCheckpoint saves the best-performing model weights based on validation accuracy.

In [ ]:

# Configure EarlyStopping Callback

early_stop = EarlyStopping(

    # Monitor validation loss
    monitor="val_loss",

    # Stop after 3 epochs without improvement
    patience=3,

    # Restore best model weights
    restore_best_weights=True
)


# Configure ModelCheckpoint Callback
checkpoint = ModelCheckpoint(

    # File name for saved model
    "best_lstm_model_partB.h5",

    # Monitor validation accuracy
    monitor="val_accuracy",

    # Save only best-performing model
    save_best_only=True,

    # Display checkpoint updates
    verbose=1
)

print("Callbacks configured successfully.")

## Insights

Training callbacks were configured to improve training stability, reduce overfitting risk, and preserve the best-performing LSTM model during training.

The EarlyStopping callback monitors validation loss and automatically stops training if validation performance does not improve for three consecutive epochs. This helps prevent unnecessary training and reduces the risk of overfitting.

Additionally, the `restore_best_weights=True` setting ensures that the final model retains the best-performing weights observed during training rather than the weights from the final epoch.

The ModelCheckpoint callback saves the best-performing model based on validation accuracy, allowing the strongest version of the improved LSTM model to be preserved for later evaluation, reproducibility, and model comparison experiments.

## 16.2 Train LSTM Model

The LSTM model is trained using the translated and padded review sequences together with the configured training callbacks.

Training and validation performance are monitored across epochs to evaluate whether the enhanced LSTM architecture improves sentiment classification capability compared to the earlier baseline SimpleRNN model.

In [ ]:

# Train LSTM Model

history2 = improved_model.fit(

    # Training input sequences
    X_train,

    # Training sentiment labels
    y_train,

    # Maximum training epochs
    epochs=15,

    # Batch size
    batch_size=32,

    # Validation split from training set
    validation_split=0.2,

    # Training callbacks
    callbacks=[early_stop, checkpoint]
)

## Insights

The improved LSTM model demonstrated strong learning performance during training and significantly outperformed the earlier baseline SimpleRNN model.

Training accuracy and validation accuracy both improved steadily across epochs, while validation loss decreased substantially during the training process. The best validation accuracy achieved was approximately 89.6%, representing a clear improvement over the baseline recurrent architecture.

The ModelCheckpoint callback successfully preserved the best-performing model weights during training, while the EarlyStopping callback helped reduce unnecessary training and limited overfitting risk.

Although training accuracy eventually exceeded 98%, the validation accuracy improved more gradually and stabilised near 90%, suggesting that mild overfitting began to emerge during later training epochs. However, the combination of dropout regularisation and callbacks helped maintain strong generalisation performance.

Overall, the results suggest that the LSTM architecture was substantially more effective than the baseline SimpleRNN model at learning contextual sentiment relationships from the translated pizza review dataset.

## 16.3 Best Validation Performance Achieved During Training

In [ ]:
# Display highest validation accuracy achieved
print(
    f"Best Validation Accuracy Achieved: "
    f"{max(history2.history['val_accuracy']):.4f}"
)

# Display lowest validation loss achieved
print(
    f"Lowest Validation Loss Achieved: "
    f"{min(history2.history['val_loss']):.4f}"
)

## Insights

The LSTM model achieved a best validation accuracy of approximately 89.6% and a lowest validation loss of approximately 0.4237 during training.

These results indicate that the improved LSTM architecture learned meaningful contextual sentiment patterns from the translated pizza review dataset while maintaining strong generalisation performance on unseen validation data.

Compared to the earlier baseline SimpleRNN model, the improved LSTM achieved substantially stronger validation performance, suggesting that the gated recurrent architecture was more effective at capturing sequential contextual relationships in multilingual sentiment analysis.

The relatively low validation loss also indicates that the model predictions became increasingly confident and consistent throughout training.

## 16.4 Evaluate LSTM Model

The LSTM model is evaluated using the unseen testing dataset to measure its final generalisation performance on multilingual sentiment classification.

The resulting testing metrics are compared against the earlier baseline SimpleRNN model to determine whether the architectural improvements successfully enhanced sentiment prediction capability.

In [ ]:
# Evaluate LSTM Model
# Evaluate model on unseen testing dataset
loss2, acc2 = improved_model.evaluate(
    X_test,
    y_test
)

# Display testing loss
print(
    f"Improved LSTM Loss: {loss2:.4f}"
)

# Display testing accuracy
print(
    f"Improved LSTM Accuracy: {acc2:.4f}"
)

## Insights

The LSTM model achieved a test accuracy of approximately 90.38% with a test loss of approximately 0.2956 on the unseen testing dataset.

Compared to the earlier baseline SimpleRNN model, the improved LSTM architecture demonstrated substantially stronger sentiment classification performance, indicating that the gated recurrent learning mechanism was more effective at capturing contextual sentiment relationships within the translated review dataset.

The testing accuracy is also very close to the best validation accuracy achieved during training, suggesting that the model maintained strong generalisation capability without severe overfitting.

Additionally, the relatively low testing loss indicates that the LSTM model produced more confident and consistent sentiment predictions across the three sentiment classes.

Overall, the results suggest that the enhanced LSTM architecture significantly improved multilingual sentiment classification performance compared to the baseline recurrent neural network model.

# Step 17: GRU Model Architecture Experiment

A Gated Recurrent Unit (GRU) model is developed as an alternative gated recurrent architecture to compare against the improved LSTM model for multilingual sentiment classification performance.

In [ ]:

# Build GRU Model

gru_model = Sequential(
    name="GRU_Model"
)


# Embedding Layer
gru_model.add(

    Embedding(

        # Vocabulary size
        max_words,

        # Embedding dimension
        64,

        # Input sequence length
        input_length=max_length
    )
)



# GRU Layer
gru_model.add(

    GRU(

        # Number of GRU units
        128
    )
)


# Dropout Regularisation
gru_model.add(

    Dropout(0.5)
)


# Dense Hidden Layer
gru_model.add(

    Dense(
        64,
        activation="relu"
    )
)


# Output Layer
gru_model.add(

    Dense(
        3,
        activation="softmax"
    )
)


# Compile GRU Model
gru_model.compile(

    loss="sparse_categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)

# Display model summary
gru_model.summary()

## Insights

The GRU model was developed as an alternative gated recurrent architecture for multilingual sentiment classification.

The model uses an enhanced architecture consisting of:

- a larger embedding layer
- 128 GRU units
- dropout regularisation
- an additional Dense hidden layer

Compared to LSTM networks, GRU architectures contain fewer internal gating mechanisms and therefore require fewer trainable parameters while still maintaining strong sequential contextual learning capability.

The GRU model contains fewer parameters than the improved LSTM model, making it computationally simpler and potentially faster to train while still remaining highly effective for sentiment classification tasks.

Importantly, the GRU architecture was intentionally designed to remain structurally similar to the improved LSTM model so that performance differences can be compared more fairly between the two gated recurrent approaches.

This experiment therefore allows direct comparison between LSTM and GRU architectures to evaluate which recurrent approach produces stronger multilingual sentiment classification performance on the pizza review dataset.

In [ ]:
# Display GRU model input shape
print(gru_model.input_shape)

# Display GRU model output shape
print(gru_model.output_shape)

## Insights

The GRU model maintains the same input and output dimensions as the baseline SimpleRNN and improved LSTM models.

Each review is represented using padded sequences of length 100, while the output layer generates probability predictions across the three sentiment classes:

- Negative
- Neutral
- Positive

Maintaining consistent tensor dimensions across all recurrent architectures helps ensure fairer experimental comparison by allowing performance differences to be attributed more directly to the recurrent architecture design rather than differences in preprocessing or input representation.

The implemented GRU model also already contains multiple architectural enhancements, including larger embeddings, dropout regularisation, and an additional Dense hidden layer, making it a suitable advanced recurrent architecture for comparison against the improved LSTM model.

## 17.1 Model Configuration Justification

Several neural network hyperparameters were selected based on balancing learning capability, computational efficiency, and model generalisation performance.

A batch size of 32 was used because it provides relatively stable gradient updates while maintaining efficient training performance.

Embedding dimensions of 64 were selected to allow richer semantic word representations compared to smaller embedding sizes, helping the model capture more meaningful contextual relationships between words.

The GRU layer used 128 recurrent units to improve the model’s ability to capture sequential and contextual sentiment patterns within the review text.

A dropout rate of 0.5 was applied to reduce overfitting risk and improve generalisation performance by preventing excessive reliance on specific neurons during training.

Training was performed for up to 15 epochs, while EarlyStopping was used to automatically prevent excessive overtraining once validation performance stopped improving.

## 17.2 Configure GRU Callbacks

In [ ]:
# Configure GRU EarlyStopping Callback

gru_early_stop = EarlyStopping(

    # Monitor validation loss
    monitor="val_loss",

    # Stop after 3 epochs without improvement
    patience=3,

    # Restore best-performing weights
    restore_best_weights=True
)



# Configure GRU ModelCheckpoint Callback
gru_checkpoint = ModelCheckpoint(

    # File name for saved GRU model
    "best_gru_model_partB.h5",

    # Monitor validation accuracy
    monitor="val_accuracy",

    # Save only best-performing model
    save_best_only=True,

    # Display checkpoint updates
    verbose=1
)

## Insights

The same callback strategy used for the improved LSTM model was also applied to the GRU model to maintain fair experimental comparison between recurrent architectures.

EarlyStopping helps reduce overfitting risk by monitoring validation loss and automatically stopping training when validation performance no longer improves, while ModelCheckpoint preserves the best-performing GRU model weights during training.

Using consistent training configurations across both LSTM and GRU experiments helps ensure that performance differences can be attributed more directly to architectural differences rather than variations in training strategy or regularisation behaviour.

This controlled experimental setup improves the reliability and fairness of the recurrent architecture comparison process.

## 17.3 Train GRU Model

The GRU model is trained using the translated and padded review sequences together with the configured callbacks.

Training and validation performance are monitored across epochs to evaluate whether the GRU architecture can achieve competitive multilingual sentiment classification performance compared to the improved LSTM model.

In [ ]:
# Train GRU Model

gru_history = gru_model.fit(

    # Training input sequences
    X_train,

    # Training sentiment labels
    y_train,

    # Maximum number of epochs
    epochs=15,

    # Batch size
    batch_size=32,

    # Validation split
    validation_split=0.2,

    # GRU callbacks
    callbacks=[
        gru_early_stop,
        gru_checkpoint
    ]
)

## Insights

The GRU model demonstrated strong learning performance during training and achieved competitive validation accuracy for multilingual sentiment classification.

Training accuracy and validation accuracy improved steadily across epochs, indicating that the GRU architecture successfully learned meaningful sequential sentiment patterns from the translated pizza review dataset.

The best validation accuracy achieved was approximately 88.0%, while the lowest validation loss achieved was approximately 0.4566. These results indicate that the GRU model was able to generalise reasonably well to unseen validation data.

Compared to the improved LSTM model, the GRU architecture achieved slightly lower validation performance. However, GRU networks are computationally simpler and contain fewer trainable parameters, making them an efficient alternative recurrent architecture for sentiment classification tasks.

During later epochs, training accuracy continued increasing while validation performance stabilised, suggesting that mild overfitting began to emerge. Nevertheless, the use of dropout regularisation and training callbacks helped control excessive overfitting and maintain relatively stable validation performance.

## 17.4 Best GRU Validation Performance Achieved During Training

In [ ]:
# Display best validation accuracy achieved
print(
    f"Best GRU Validation Accuracy: "
    f"{max(gru_history.history['val_accuracy']):.4f}"
)

# Display lowest validation loss achieved
print(
    f"Lowest GRU Validation Loss: "
    f"{min(gru_history.history['val_loss']):.4f}"
)

## Insights

The GRU model achieved a best validation accuracy of approximately 88.0% and a lowest validation loss of approximately 0.4566 during training.

These results indicate that the GRU architecture successfully learned meaningful sequential sentiment patterns from the translated pizza review dataset while maintaining relatively stable validation performance.

Compared to the LSTM model, the GRU architecture achieved slightly lower validation accuracy. However, GRU networks are computationally simpler and contain fewer trainable parameters, making them an efficient alternative recurrent architecture for multilingual sentiment classification tasks.

The validation metrics also suggest that the GRU model maintained reasonable generalisation capability, although mild overfitting tendencies began to appear during later training epochs.

## 17.5 Evaluate GRU Model

The GRU model is evaluated using the unseen testing dataset to measure its final multilingual sentiment classification performance.

The resulting testing metrics are compared against the baseline SimpleRNN and LSTM architectures to evaluate the effectiveness of GRU-based recurrent learning.

In [ ]:

# Evaluate GRU Model

# Evaluate GRU model on testing dataset
gru_loss, gru_acc = gru_model.evaluate(
    X_test,
    y_test
)

# Display testing loss
print(
    f"GRU Model Loss: {gru_loss:.4f}"
)

# Display testing accuracy
print(
    f"GRU Model Accuracy: {gru_acc:.4f}"
)

## Insights

The GRU model achieved a test accuracy of approximately 89.74% with a test loss of approximately 0.2933 on the unseen testing dataset.

Compared to the baseline SimpleRNN model, the GRU architecture demonstrated substantially stronger multilingual sentiment classification performance, indicating that gated recurrent learning mechanisms were more effective at capturing contextual sentiment relationships within the translated review dataset.

Although the LSTM model achieved slightly stronger testing performance overall, the GRU model still produced highly competitive results while using fewer trainable parameters and a computationally simpler recurrent architecture.

These findings suggest that GRU networks can provide strong multilingual sentiment classification performance while maintaining lower computational complexity compared to LSTM architectures.

The relatively low testing loss and strong testing accuracy also indicate that the GRU model learned meaningful sequential sentiment patterns from the translated pizza review dataset and maintained stable prediction performance across the three sentiment classes.

# Step 18: Training Visualisation

Training curves are plotted to analyse learning behaviour, convergence patterns, and potential overfitting across the recurrent neural network models.

## 18.1 Baseline SimpleRNN Training Curves

In [ ]:

# Plot Baseline SimpleRNN Training Curves

plt.figure(figsize=(12,5))


# Accuracy Curves
plt.subplot(1,2,1)

# Training accuracy
plt.plot(
    history.history["accuracy"],
    label="Train Accuracy"
)

# Validation accuracy
plt.plot(
    history.history["val_accuracy"],
    label="Validation Accuracy"
)

plt.title("Baseline SimpleRNN Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()




# Loss Curves
plt.subplot(1,2,2)

# Training loss
plt.plot(
    history.history["loss"],
    label="Train Loss"
)

# Validation loss
plt.plot(
    history.history["val_loss"],
    label="Validation Loss"
)

plt.title("Baseline SimpleRNN Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.tight_layout()
plt.show()

## Insights

The Baseline SimpleRNN training curves show that both training accuracy and validation accuracy improved steadily across epochs, indicating that the recurrent neural network successfully learned meaningful sentiment patterns from the translated review dataset.

At the same time, both training loss and validation loss decreased consistently throughout training, suggesting stable optimisation and progressively improving prediction performance.

During later epochs, training accuracy increased more rapidly than validation accuracy, indicating that mild overfitting began to emerge as the model continued fitting the training data more aggressively.

However, validation loss continued decreasing gradually, suggesting that overfitting remained reasonably controlled during the short training process.

Overall, the baseline SimpleRNN model demonstrated effective sequential learning capability and achieved reasonable sentiment classification performance, although its final performance remained lower than the more advanced LSTM and GRU architectures explored later in the project.

### 18.2 LSTM Training Curves

In [ ]:
plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.plot(history2.history["accuracy"], label="Train Accuracy")
plt.plot(history2.history["val_accuracy"], label="Validation Accuracy")
plt.title("LSTM Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

plt.subplot(1,2,2)
plt.plot(history2.history["loss"], label="Train Loss")
plt.plot(history2.history["val_loss"], label="Validation Loss")
plt.title("LSTM Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.tight_layout()
plt.show()

## Insights

The LSTM model training curves show that both training accuracy and validation accuracy improved steadily across epochs, indicating that the model successfully learned meaningful sequential sentiment patterns from the translated pizza review dataset.

Training loss and validation loss both decreased significantly during the earlier training epochs, suggesting effective optimisation and progressively improving prediction performance throughout the learning process.

As training progressed, training accuracy continued increasing while validation accuracy improved more gradually and eventually stabilised. At the same time, training loss continued decreasing while validation loss plateaued slightly during later epochs. This indicates that mild overfitting began to emerge as the model learned the training data more aggressively.

However, the validation curves remained relatively stable overall, suggesting that the combination of dropout regularisation and training callbacks helped control excessive overfitting and maintain good generalisation performance.

Compared to the baseline SimpleRNN model, the LSTM model achieved substantially stronger overall validation and testing performance, demonstrating the effectiveness of gated recurrent learning for multilingual sentiment classification tasks.

### 18.3 GRU Training Curves

In [ ]:
plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.plot(gru_history.history["accuracy"], label="Train Accuracy")
plt.plot(gru_history.history["val_accuracy"], label="Validation Accuracy")
plt.title("GRU Model Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

plt.subplot(1,2,2)
plt.plot(gru_history.history["loss"], label="Train Loss")
plt.plot(gru_history.history["val_loss"], label="Validation Loss")
plt.title("GRU Model Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.tight_layout()
plt.show()

## Insights

The GRU model training curves show that both training accuracy and validation accuracy improved steadily during the earlier training epochs, indicating that the GRU architecture successfully learned meaningful sequential sentiment patterns from the translated pizza review dataset.

Training loss and validation loss both decreased significantly during the early stages of training, suggesting effective optimisation and progressively improving prediction performance.

However, after approximately Epoch 5, validation loss began increasing while training loss continued decreasing. At the same time, training accuracy continued improving while validation accuracy stabilised around the high-80% range. This indicates that mild overfitting began to emerge during the later training epochs as the model increasingly memorised training data patterns.

Despite this, the validation accuracy remained relatively stable overall, suggesting that the GRU model still maintained good generalisation capability on unseen validation data.

Compared to the baseline SimpleRNN model, the GRU architecture achieved substantially stronger learning performance. Although the GRU model achieved slightly lower validation accuracy than the improved LSTM model during training, it later demonstrated highly competitive testing performance while maintaining lower computational complexity and fewer trainable parameters.

# Step 19: Model Comparison

The baseline SimpleRNN, improved LSTM, and GRU models are compared using test accuracy, test loss, architecture, and parameter count.

In [ ]:
comparison_df = pd.DataFrame({
    "Model": [
        "Baseline SimpleRNN",
        "LSTM Model",
        "GRU Model"
    ],

    "Test Accuracy": [
        acc,
        acc2,
        gru_acc
    ],

    "Test Loss": [
        loss,
        loss2,
        gru_loss
    ],

    "Architecture": [
        "Embedding + SimpleRNN",
        "Embedding + LSTM + Dropout + Dense",
        "Embedding + GRU + Dropout + Dense"
    ],

    "Parameters": [
        model.count_params(),
        improved_model.count_params(),
        gru_model.count_params()
    ]
})

comparison_df["Test Accuracy"] = comparison_df["Test Accuracy"].round(4)
comparison_df["Test Loss"] = comparison_df["Test Loss"].round(4)

comparison_df

## Insights

The comparison table shows that the Baseline SimpleRNN model achieved the lowest test accuracy of 76.92% and the highest test loss of 0.6635. This indicates that a simple recurrent architecture has limited ability to capture more complex contextual sentiment patterns compared to gated recurrent architectures.

The LSTM model achieved the highest test accuracy of 90.38%, showing the strongest overall classification performance on the unseen testing dataset. This suggests that the LSTM architecture was highly effective at learning contextual sentiment relationships from the translated pizza review dataset.

The GRU model achieved a very competitive test accuracy of 89.74% and the lowest test loss of 0.2933 while using fewer parameters than the LSTM model. This indicates that the GRU architecture provided strong predictive performance with slightly better parameter efficiency.

Overall, the LSTM model performed best based on test accuracy, while the GRU model offered the best balance between strong performance, lower test loss, and reduced architectural complexity. The baseline SimpleRNN model remains useful as an initial benchmark, but the gated recurrent architectures clearly provided stronger sentiment classification performance.

## 19.2 Visualise Test Accuracy Comparison

In [ ]:
plt.figure(figsize=(8,5))

ax = sns.barplot(
    x="Model",
    y="Test Accuracy",
    data=comparison_df
)

for i in ax.containers:
    ax.bar_label(i, fmt="%.4f")

plt.title("Model Accuracy Comparison")
plt.ylabel("Accuracy")

plt.ylim(0,1)

plt.show()

## Insights

The test accuracy comparison chart clearly shows that the gated recurrent architectures substantially outperformed the baseline SimpleRNN model on multilingual sentiment classification.

The Baseline SimpleRNN model achieved the lowest testing accuracy, indicating that simple recurrent architectures have more limited capability for capturing complex contextual sentiment patterns within sequential review text.

Both the LSTM and GRU models achieved significantly stronger performance due to their gated recurrent mechanisms, which improve the preservation of contextual information during sequence learning.

Among the tested architectures, the LSTM model achieved the highest testing accuracy of approximately 90.38%, slightly outperforming the GRU model. This suggests that the LSTM architecture was most effective at capturing contextual sentiment relationships within the translated pizza review dataset.

However, the GRU model still achieved highly competitive performance while maintaining lower architectural complexity and fewer trainable parameters, making it a computationally efficient alternative for multilingual sentiment classification tasks.

## 19.3 Visualise Test Loss Comparison

In [ ]:
plt.figure(figsize=(8,5))

ax = sns.barplot(
    x="Model",
    y="Test Loss",
    data=comparison_df
)

for i in ax.containers:
    ax.bar_label(i, fmt="%.4f")

plt.title("Model Loss Comparison")
plt.ylabel("Loss")



plt.show()

## Insights

The test loss comparison chart shows that both gated recurrent architectures substantially outperformed the baseline SimpleRNN model in terms of prediction loss.

The Baseline SimpleRNN model produced the highest testing loss, indicating weaker sequential learning capability and less stable prediction performance on multilingual sentiment classification.

Among the evaluated architectures, the GRU model achieved the lowest testing loss of approximately 0.2933, slightly outperforming the LSTM model. This suggests that the GRU architecture produced highly stable and confident prediction probabilities while maintaining strong classification performance.

Although the LSTM model achieved slightly higher testing accuracy overall, the GRU model demonstrated comparable predictive capability with lower testing loss and fewer trainable parameters. This highlights the strong efficiency of GRU architectures for multilingual sentiment classification tasks.

Overall, both LSTM and GRU architectures significantly improved sentiment classification performance compared to the baseline recurrent model, demonstrating the effectiveness of gated recurrent learning for sequential text analysis.

## 19.4 Visualise Parameters Comparison

In [ ]:
plt.figure(figsize=(8,5))

ax = sns.barplot(
    x="Model",
    y="Parameters",
    data=comparison_df
)

for i in ax.containers:
    ax.bar_label(i, fmt="%d")

plt.title("Model Parameter Comparison")
plt.ylabel("Number of Parameters")



plt.show()

## Insights

The parameter comparison chart highlights the differences in architectural complexity across the three recurrent neural network models.

The Baseline SimpleRNN model contained the fewest trainable parameters, making it the simplest architecture computationally. However, it also achieved the weakest sentiment classification performance, suggesting that the lower model complexity limited its ability to capture more complex contextual patterns within multilingual review text.

The LSTM model contained the highest number of trainable parameters due to its larger embedding dimensions, gated recurrent structure, dropout regularisation, and additional Dense layers. This increased complexity contributed to strong learning capability and the highest overall testing accuracy.

The GRU model contained slightly fewer trainable parameters than the LSTM model while still achieving highly competitive testing performance. This suggests that the GRU architecture provided strong parameter efficiency and achieved excellent sentiment classification capability with lower computational complexity.

Overall, the parameter comparison demonstrates the trade-off between model complexity, computational efficiency, and predictive performance across different recurrent neural network architectures.

## 19.5 Best Model Selection

In [ ]:
best_model = comparison_df.loc[
    comparison_df["Test Accuracy"].idxmax()
]

print("Best Performing Model:\n")
print(f"Model: {best_model['Model']}")
print(f"Accuracy: {best_model['Test Accuracy']:.4f}")
print(f"Loss: {best_model['Test Loss']:.4f}")
print(f"Architecture: {best_model['Architecture']}")
print(f"Parameters: {best_model['Parameters']}")

## Insights

Based on the overall experimental results, the LSTM model was selected as the best-performing architecture for multilingual sentiment classification.

The LSTM model achieved the highest testing accuracy of approximately 90.38%, indicating the strongest overall classification performance on unseen review data. In addition, the LSTM model maintained relatively low testing loss, suggesting stable and confident prediction capability across the three sentiment classes.

Although the GRU model achieved slightly lower testing loss and used fewer trainable parameters, the LSTM architecture ultimately produced the highest overall testing accuracy, which was prioritised as the primary evaluation metric for this sentiment classification task.

These findings suggest that the LSTM architecture provided the strongest balance between predictive performance, contextual learning capability, and generalisation performance within this project.

The selected LSTM model will therefore be used as the final model for further evaluation, prediction analysis, and model persistence.

## 19.6 Accuracy Improvement from Baseline SimpleRNN to Final LSTM Model

In [ ]:
# Calculate percentage accuracy improvement (relative)
improvement = ((acc2 - acc) / acc) * 100

print(
    f"Accuracy Improvement from "
    f"Baseline SimpleRNN to LSTM Model: "
    f"{improvement:.2f}%"
)

Relative percentage improvement was calculated to measure how much the LSTM model improved sentiment classification performance relative to the Baseline SimpleRNN model.

Unlike absolute accuracy increase, relative improvement helps quantify the strength of the performance gain compared to the original baseline model performance, providing clearer insight into the effectiveness of the improved recurrent neural network architecture.

## Insights

The LSTM model improved testing accuracy from 76.92% to 90.38%, representing an absolute improvement of 13.46 percentage points and a relative accuracy improvement of approximately 17.50% compared to the Baseline SimpleRNN model.

This substantial improvement demonstrates the effectiveness of LSTM architectures in learning long-term sequential dependencies and contextual sentiment patterns within multilingual review text.

Unlike the SimpleRNN architecture, the LSTM model uses gated memory mechanisms that help preserve important contextual information across longer text sequences, improving sentiment classification performance and generalisation capability.

## 19.7 Computational Considerations

The LSTM model contains substantially more trainable parameters than the Baseline SimpleRNN model because it uses larger embedding dimensions, increased recurrent units, dropout regularisation, and an additional Dense layer.

This demonstrates a common trade-off in deep learning: more complex architectures may improve predictive performance, but they also require greater computational resources, higher memory usage, and longer training time.

The GRU model provides an important comparison because it achieved highly competitive sentiment classification performance while using slightly fewer trainable parameters than the LSTM model.

Interestingly, although the GRU architecture was computationally simpler than the LSTM model, it still achieved testing performance very close to the LSTM architecture. This suggests that GRU networks can provide an effective balance between predictive performance and computational efficiency for multilingual sentiment classification tasks.

# Step 20.1 Sample Predictions Using Final LSTM Model

Sample reviews are used to demonstrate how the final selected LSTM model predicts sentiment labels and confidence scores.

In [ ]:

# Generate Sample Predictions Using Final LSTM Model

# Randomly select sample reviews
sample_texts = (
    df["Review_Translated"]
    .sample(5, random_state=42)
    .values
)

# Clean sample reviews
sample_cleaned = [
    clean_text(text)
    for text in sample_texts
]

# Convert text into tokenizer sequences
sample_seq = tokenizer.texts_to_sequences(
    sample_cleaned
)

# Pad sequences to fixed length
sample_pad = pad_sequences(
    sample_seq,
    maxlen=max_length
)

# Generate predictions using final LSTM model
preds = improved_model.predict(sample_pad)

# Get predicted class indices
pred_labels = np.argmax(preds, axis=1)

# Convert numerical labels back to sentiment names
decoded_labels = encoder.inverse_transform(
    pred_labels
)

# Display prediction results
for i in range(len(sample_texts)):

    print("Review:", sample_texts[i])

    print(
        "Predicted Sentiment:",
        decoded_labels[i]
    )

    print(
        "Prediction Confidence:",
        f"{np.max(preds[i]):.4f}"
    )

    print("-"*50)

## Insights

The sample predictions demonstrate how the final LSTM model classifies unseen multilingual review text into Negative, Neutral, and Positive sentiment categories.

Most reviews containing strongly positive wording such as “tasty”, “satisfying”, “perfectly”, and “definitely order again” were confidently predicted as Positive sentiment with very high confidence scores close to 1.0.

The shorter review “Pretty okay” was classified as Neutral sentiment with a noticeably lower confidence score compared to the strongly positive reviews. This suggests that the model recognised the more moderate and less emotionally intense nature of the review while also reflecting greater uncertainty due to the limited contextual information available.

The prediction results indicate that the LSTM model successfully learned meaningful contextual sentiment patterns from the translated pizza review dataset and was able to generate stable sentiment predictions for unseen review samples.

Overall, the sample predictions demonstrate the effectiveness of the final LSTM model for multilingual sentiment classification while also highlighting how shorter or more ambiguous reviews may naturally produce lower prediction confidence.

## 20.2 Detailed Sentiment Probability Analysis

In [ ]:
for i in range(len(sample_texts)):

    print("Review:", sample_texts[i])
    print("Predicted Sentiment:", decoded_labels[i])
    print(f"Negative Probability: {preds[i][0]:.4f}")
    print(f"Neutral Probability: {preds[i][1]:.4f}")
    print(f"Positive Probability: {preds[i][2]:.4f}")
    print("-"*50)

## Insights

The detailed sentiment probability distributions provide deeper insight into how the final LSTM model evaluates competing sentiment classes during prediction.

Reviews containing strong positive wording produced highly concentrated Positive probabilities close to 1.0. For example, reviews containing phrases such as “very tasty”, “satisfying”, and “definitely order again” generated extremely high Positive probabilities while assigning near-zero probabilities to the Negative and Neutral classes.

The review “Pretty okay” was predicted as Neutral sentiment with a high Neutral probability of approximately 94.9%. This suggests that the model was able to recognise the moderate and less emotionally intense tone of the review rather than incorrectly forcing a strongly positive or negative classification.

The probability distributions also demonstrate that the LSTM model learned meaningful contextual sentiment patterns from the translated pizza review dataset. When clear sentiment cues were present, the model produced highly confident predictions with strongly dominant probability distributions.

Overall, the probability analysis shows that the final LSTM model was capable of generating stable and confident multilingual sentiment predictions while appropriately distinguishing between strongly emotional and more moderate review expressions.

# Step 21. Additional Experiments for Stronger Evaluation

The assignment asks whether other languages should be kept or discarded and whether the given data can be engineered to produce more training data.

Therefore, additional experiments are conducted:

- Original training data vs sentence-level augmented data
- English-only reviews vs English and translated Malay reviews
- Class weight experiment

These experiments justify preprocessing and modelling decisions using actual evaluation results.

---

## 21.1 Helper Functions for Fair Experiments

The same LSTM architecture and evaluation metrics are used for the additional experiments.

The LSTM architecture was selected because it achieved the strongest overall sentiment classification performance during the earlier model comparison experiments.

Using the same LSTM architecture across all additional experiments helps maintain consistency with the final selected model while ensuring fair comparison between different preprocessing and data engineering strategies.

Importantly, the objective of these experiments is not to identify a new final model, but rather to evaluate the relative impact of different preprocessing strategies under a controlled and consistent experimental framework.

For consistency with the revised workflow, tokenization is performed before the train-test split inside the experiment helper function.

The assignment requires additional justification for:

1. Whether other languages should be retained or removed
2. Whether data augmentation can improve performance
3. Whether class balancing strategies are useful

To ensure fair comparison, all experiments use:

- the same LSTM architecture
- the same tokenization workflow
- the same train-test split strategy
- the same evaluation metrics

This ensures that performance differences are mainly caused by the preprocessing strategy being tested.

In [ ]:

# Additional Experiments for Stronger Evaluation

# Helper Function:
# Build Standardized LSTM Experiment Model

def build_lstm_experiment_model(
    vocab_size=5000,
    max_length=100
):

    # Create Sequential LSTM model
    model_exp = Sequential(
        name="LSTM_Experiment_Model"
    )

   
    # Embedding Layer
 
    # Converts integer word indices into dense vectors

    model_exp.add(

        Embedding(
            vocab_size,
            64,
            input_length=max_length
        )
    )


    # LSTM Layer
 
    # Learns sequential relationships within review text

    model_exp.add(

        LSTM(128)
    )


    # Dropout Layer

    # Reduces overfitting by randomly disabling neurons
    # during training

    model_exp.add(

        Dropout(0.5)
    )


    # Dense Hidden Layer

    # Learns higher-level sentiment representations

    model_exp.add(

        Dense(
            64,
            activation="relu"
        )
    )

  
    # Output Layer

    # Produces probabilities for:
    # Negative / Neutral / Positive

    model_exp.add(

        Dense(
            3,
            activation="softmax"
        )
    )


    # Compile Model


    model_exp.compile(

        loss="sparse_categorical_crossentropy",

        optimizer="adam",

        metrics=["accuracy"]
    )

    # Return compiled model
    return model_exp



# Helper Function:
# Train and Evaluate Experiment


def train_and_evaluate_experiment(

    # Input experiment dataset
    experiment_df,

    # Experiment name for printing results
    experiment_name,

    # Text column used for training
    text_col="Clean_Review",

    # Vocabulary size
    max_words=5000,

    # Maximum sequence length
    max_length=100
):


    # Encode Sentiment Labels

    # Convert:
    # Negative / Neutral / Positive
    # into numerical labels

    label_encoder_exp = LabelEncoder()

    y_exp = label_encoder_exp.fit_transform(
        experiment_df["Sentiment"]
    )


    # Create Tokenizer

    tokenizer_exp = Tokenizer(
        num_words=max_words,
        oov_token="<OOV>"
    )


    # Fit Tokenizer on Experiment Text Dataset

    tokenizer_exp.fit_on_texts(
        experiment_df[text_col]
    )


    # Convert Text into Integer Sequences

    X_seq_exp = tokenizer_exp.texts_to_sequences(
        experiment_df[text_col]
    )


    # Pad Sequences to Fixed Length

    X_pad_exp = pad_sequences(
        X_seq_exp,
        maxlen=max_length
    )


    # Train-Test Split
    # Stratified split preserves class distribution

    X_train_exp, X_test_exp, y_train_exp, y_test_exp = (

        train_test_split(

            X_pad_exp,

            y_exp,

            test_size=0.2,

            random_state=42,

            stratify=y_exp
        )
    )


    # Build Standardized LSTM Model

    model_exp = build_lstm_experiment_model(

        vocab_size=max_words,

        max_length=max_length
    )


    # EarlyStopping Callback

    # Stops training if validation loss no longer improves

    early_stop_exp = EarlyStopping(

        monitor="val_loss",

        patience=3,

        restore_best_weights=True
    )


    # Train LSTM Model

    history_exp = model_exp.fit(

        X_train_exp,

        y_train_exp,

        epochs=15,

        batch_size=32,

        validation_split=0.2,

        callbacks=[early_stop_exp],

        verbose=1
    )


    # Generate Predictions

    y_pred_prob_exp = model_exp.predict(
        X_test_exp,
        verbose=0
    )


    # Convert Probabilities into Predicted Labels

    y_pred_exp = np.argmax(
        y_pred_prob_exp,
        axis=1
    )


    # Calculate Evaluation Metrics

    accuracy_exp = accuracy_score(
        y_test_exp,
        y_pred_exp
    )

    macro_f1_exp = f1_score(
        y_test_exp,
        y_pred_exp,
        average="macro"
    )


    # Display Experiment Results
    print("="*60)
    print("Experiment:", experiment_name)
    print("="*60)

    print(f"Accuracy: {accuracy_exp:.4f}")

    print(
        f"Macro F1-score: "
        f"{macro_f1_exp:.4f}"
    )

    print()


    # Display Classification Report

    print(

        classification_report(

            y_test_exp,

            y_pred_exp,

            target_names=label_encoder_exp.classes_
        )
    )


    # Return Results Dictionary

    # Useful for later comparison and visualization

    return {

        "Experiment": experiment_name,

        "Accuracy": accuracy_exp,

        "Macro F1-score": macro_f1_exp,

        "Model": model_exp,

        "History": history_exp,

        "Tokenizer": tokenizer_exp,

        "Label Encoder": label_encoder_exp,

        "Y Test": y_test_exp,

        "Y Pred": y_pred_exp
    }



# Confirm Helper Functions Created Successfully


print(
    "Experiment helper functions configured successfully."
)

## Insights

Helper functions were developed to standardise the experimental workflow across all additional sentiment classification experiments.

The same LSTM architecture, tokenization workflow, train-test split strategy, training configuration, and evaluation metrics were consistently applied throughout the experiments. This helps ensure that performance differences are more likely caused by the preprocessing or data engineering strategy being tested rather than inconsistencies in model configuration.

The LSTM architecture was selected because it previously achieved the strongest overall sentiment classification performance during the earlier model comparison experiments. Using the same architecture throughout the additional experiments also improves consistency with the final selected model.

Using reusable helper functions improves reproducibility, reduces duplicated code, and supports fairer experimental comparison across different preprocessing strategies.

Additionally, the inclusion of macro F1-score alongside accuracy provides a more balanced evaluation of multi-class sentiment classification performance, especially when analysing class-specific behaviour across Negative, Neutral, and Positive reviews.

Overall, this experimental framework provides a more reliable and controlled methodology for evaluating the impact of multilingual retention, sentence-level augmentation, and class balancing strategies on sentiment classification performance.

### 21.1.1 Train-Test Controlled Experiment Function

In [ ]:

# Train-Test Controlled Experiment Function

# This function allows externally controlled train-test
# datasets so augmentation can be applied only to the
# training data.


def train_and_evaluate_custom_split(

    train_data,
    test_data,
    experiment_name,
    text_col="Clean_Review",
    max_words=5000,
    max_length=100

):

 
    # Encode Sentiment Labels

    label_encoder_exp = LabelEncoder()

    y_train = label_encoder_exp.fit_transform(
        train_data["Sentiment"]
    )

    y_test = label_encoder_exp.transform(
        test_data["Sentiment"]
    )



    # Fit Tokenizer on Training Data Only

    tokenizer_exp = Tokenizer(
        num_words=max_words,
        oov_token="<OOV>"
    )

    tokenizer_exp.fit_on_texts(
        train_data[text_col]
    )



    # Convert Text to Sequences

    X_train_seq = tokenizer_exp.texts_to_sequences(
        train_data[text_col]
    )

    X_test_seq = tokenizer_exp.texts_to_sequences(
        test_data[text_col]
    )



    # Pad Sequences

    X_train_pad = pad_sequences(
        X_train_seq,
        maxlen=max_length
    )

    X_test_pad = pad_sequences(
        X_test_seq,
        maxlen=max_length
    )



    # Build LSTM Experiment Model

    model_exp = build_lstm_experiment_model(
        vocab_size=max_words,
        max_length=max_length
    )



    # Configure EarlyStopping

    early_stop_exp = EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True
    )



    # Train Model

    history_exp = model_exp.fit(

        X_train_pad,
        y_train,

        epochs=15,

        batch_size=32,

        validation_split=0.2,

        callbacks=[early_stop_exp],

        verbose=1
    )



    # Predict Test Set

    y_pred_prob_exp = model_exp.predict(
        X_test_pad
    )

    y_pred_exp = np.argmax(
        y_pred_prob_exp,
        axis=1
    )



    # Evaluate Performance

    accuracy_exp = accuracy_score(
        y_test,
        y_pred_exp
    )

    macro_f1_exp = f1_score(
        y_test,
        y_pred_exp,
        average="macro"
    )



    # Display Results

    print("="*60)

    print("Experiment:", experiment_name)

    print("="*60)

    print(f"Accuracy: {accuracy_exp:.4f}")

    print(f"Macro F1-score: {macro_f1_exp:.4f}")

    print()

    print(

        classification_report(

            y_test,

            y_pred_exp,

            target_names=label_encoder_exp.classes_
        )
    )


    return {

        "Experiment": experiment_name,

        "Accuracy": accuracy_exp,

        "Macro F1-score": macro_f1_exp
    }

## Insights

A separate train-test controlled experiment function was developed to ensure that data augmentation strategies are applied only to the training dataset while keeping the testing dataset completely unseen during model training.

This experimental design helps prevent data leakage and ensures fairer evaluation of augmentation effectiveness. If augmented samples were allowed to influence the testing dataset, the evaluation results could become artificially inflated and unreliable.

Unlike the earlier general experiment helper function, the tokenizer in this workflow is fitted only on the training data rather than the full dataset. This more closely reflects real-world machine learning practice, where testing data should remain isolated from training-related preprocessing steps.

The controlled workflow therefore provides a more rigorous framework for evaluating whether sentence-level augmentation genuinely improves multilingual sentiment classification performance.

## 21.2 Sentence-Level Augmentation Experiment

This experiment investigates whether sentence-level augmentation can improve sentiment classification.

Multi-sentence reviews are split into smaller sentence-level samples while preserving the original sentiment label.

This increases the amount of available training data, but may also introduce label noise because individual sentences may not always perfectly reflect the overall review sentiment.

Data augmentation is applied ONLY to the training set to prevent data leakage between training and testing. Multi-sentence reviews are split into smaller sentence samples to increase training data size.

In [ ]:

# Split Original Dataset First

# Train-test split is performed BEFORE augmentation
# to ensure augmented samples do not leak into the
# testing dataset.


train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["Sentiment"]
)



# Create Sentence-Level Augmentation Function

def create_sentence_augmented_df(input_df):

    augmented_rows = []

    for _, row in input_df.iterrows():

        # Split review into sentences
        sentences = re.split(
            r"[.!?]+",
            str(row["Review_Translated"])
        )

        for sentence in sentences:

            sentence = sentence.strip()

            # Keep meaningful sentences only
            if len(sentence.split()) >= 3:

                augmented_rows.append({

                    "Review": row["Review"],

                    "Review_Translated": sentence,

                    "Score": row["Score"],

                    "Language": row["Language"],

                    "Sentiment": row["Sentiment"],

                    "Clean_Review": clean_text(sentence)
                })

    return pd.DataFrame(augmented_rows)



# Apply Augmentation ONLY to Training Data
augmented_train_df = create_sentence_augmented_df(
    train_df
)



# Keep Original Test Set Unchanged

print(
    "Original training dataset size:",
    train_df.shape
)

print(
    "Augmented training dataset size:",
    augmented_train_df.shape
)

print(
    "Original test dataset size:",
    test_df.shape
)

## Insights

Sentence-level augmentation was applied only to the training dataset to prevent data leakage between training and testing data.

This ensures that augmented sentence samples generated from the same original review do not appear in both the training and testing sets simultaneously, helping maintain fairer model evaluation.

The augmentation process increased the number of available training samples from 621 reviews to 705 sentence-level samples by splitting multi-sentence reviews into smaller sentence-based examples while preserving the original untouched testing dataset.

This experiment aims to investigate whether increasing the amount of available training data can improve multilingual sentiment classification performance.

However, sentence-level augmentation may also introduce some label inconsistency because individual sentences may not always perfectly reflect the overall sentiment label inherited from the original review. For example, a review labelled as Neutral may contain both positive and negative sentences after splitting.

Overall, this augmentation strategy provides a controlled experimental framework for evaluating whether sentence-level data expansion improves recurrent neural network learning performance.

## 21.3 Compare Original and Sentence-Augmented Datasets

This experiment compares:
1. Original multilingual review dataset
2. Sentence-level augmented dataset

The objective is to determine whether sentence-level augmentation improves multilingual sentiment classification performance.

Both experiments use:

- identical LSTM architecture
- same tokenization workflow
- same evaluation metrics

This ensures fair comparison between datasets.

In [ ]:

# Original Dataset Experiment

original_data_result = train_and_evaluate_custom_split(

    train_df,

    test_df,

    "Original Dataset"
)



# Sentence-Augmented Dataset Experiment

augmented_data_result = train_and_evaluate_custom_split(

    augmented_train_df,

    test_df,

    "Sentence-Augmented Dataset"
)

## Insights

The original multilingual dataset and the sentence-level augmented dataset were evaluated using the same LSTM architecture and experimental workflow to ensure fair comparison.

The original dataset achieved a higher Accuracy of 0.9167 and a higher Macro F1-score of 0.9066, while the sentence-augmented dataset achieved a lower Accuracy of 0.8590 and a lower Macro F1-score of 0.8447.

Although sentence-level augmentation increased the number of available training samples, the augmented dataset produced weaker overall sentiment classification performance. This suggests that splitting full reviews into isolated sentence-level samples may have reduced important contextual information required for accurate sentiment interpretation.

The augmented dataset also demonstrated weaker class-level performance, particularly for Negative reviews. This may be caused by label inconsistency introduced during augmentation, where individual split sentences inherit the overall review sentiment label even when the sentence itself may not fully express that sentiment independently.

These findings suggest that preserving complete review-level contextual structure was more beneficial than artificially increasing training dataset size through sentence-level splitting.

Overall, the experiment demonstrates that increasing training sample quantity does not necessarily guarantee improved sentiment classification performance, especially when contextual coherence and label consistency are weakened during data augmentation.

### 21.4 Augmentation Experiment Comparison

This section summarises the performance of:

1. Original multilingual dataset
2. Sentence-level augmented dataset

The comparison focuses on:
- Accuracy
- Macro F1-score

This helps determine whether sentence-level augmentation improved multilingual sentiment classification performance.

In [ ]:
# Step 21.4 Augmentation Experiment Comparison



# Create Comparison Table
augmentation_comparison_df = pd.DataFrame([

    {
        # Original dataset experiment
        "Experiment":
            original_data_result["Experiment"],

        # Testing accuracy
        "Accuracy":
            original_data_result["Accuracy"],

        # Macro-average F1-score
        "Macro F1-score":
            original_data_result["Macro F1-score"]
    },

    {
        # Sentence-augmented dataset experiment
        "Experiment":
            augmented_data_result["Experiment"],

        # Testing accuracy
        "Accuracy":
            augmented_data_result["Accuracy"],

        # Macro-average F1-score
        "Macro F1-score":
            augmented_data_result["Macro F1-score"]
    }
])



# Round numerical metrics for cleaner presentation
augmentation_comparison_df["Accuracy"] = (
    
    augmentation_comparison_df["Accuracy"]
    .round(4)
)

augmentation_comparison_df["Macro F1-score"] = (
    
    augmentation_comparison_df["Macro F1-score"]
    .round(4)
)



# Display comparison table
augmentation_comparison_df

## Insights

The augmentation comparison table summarises the performance differences between the original multilingual dataset and the sentence-level augmented dataset.

The original dataset achieved higher Accuracy (0.9167) and Macro F1-score (0.9066) compared to the sentence-augmented dataset, which achieved an Accuracy of 0.8590 and a Macro F1-score of 0.8447.

These results indicate that preserving the original full-review structure produced stronger multilingual sentiment classification performance than sentence-level augmentation.

Although augmentation increased the number of available training samples, splitting reviews into isolated sentence-level samples likely reduced contextual coherence and introduced additional label inconsistency. This may have weakened the model’s ability to learn accurate sentiment relationships across complete reviews.

Overall, the experiment demonstrates that increasing dataset size alone does not necessarily improve deep learning performance when important contextual information and label quality are compromised during augmentation.

## 21.5 Visualise Accuracy Comparison for Augmentation Experiment

In [ ]:
# Create a figure for the accuracy comparison chart
plt.figure(figsize=(8,5))

# Create a bar chart comparing accuracy across augmentation experiments
ax = sns.barplot(
    x="Experiment",
    y="Accuracy",
    data=augmentation_comparison_df
)

# Display accuracy values above each bar
for i in ax.containers:
    ax.bar_label(i, fmt="%.4f")

# Add chart title and axis labels
plt.title("Accuracy Comparison for Augmentation Experiment")
plt.ylabel("Accuracy")
plt.xticks(rotation=0)

# Display the chart
plt.show()

## Insights

The accuracy comparison chart visually shows that the original multilingual dataset achieved higher sentiment classification accuracy than the sentence-level augmented dataset.

The original dataset achieved an Accuracy of 0.9167, while the sentence-augmented dataset achieved a lower Accuracy of 0.8590.

These results suggest that sentence-level augmentation did not improve multilingual sentiment classification performance. Although augmentation increased the number of training samples, splitting reviews into isolated sentence-level samples may have reduced contextual meaning and introduced additional label inconsistency.

The chart also demonstrates that preserving complete review-level contextual structure was more beneficial than artificially increasing dataset size through sentence-level splitting.

Overall, the visual comparison supports the conclusion that contextual coherence and label quality were more important than training sample quantity for achieving strong sentiment classification performance.

## 21.6 Visualise Macro F1-score Comparison for Augmentation Experiment

In [ ]:
# Create a figure for the Macro F1-score comparison chart
plt.figure(figsize=(8,5))

# Create a bar chart comparing Macro F1-scores
# across augmentation experiments
ax = sns.barplot(
    x="Experiment",
    y="Macro F1-score",
    data=augmentation_comparison_df
)

# Display Macro F1-score values above each bar
for i in ax.containers:
    ax.bar_label(i, fmt="%.4f")

# Add chart title and axis labels
plt.title("Macro F1-score Comparison for Augmentation Experiment")
plt.ylabel("Macro F1-score")
plt.xticks(rotation=0)

# Display the chart
plt.show()

## Insights

The Macro F1-score comparison chart visually shows that the original multilingual dataset achieved stronger balanced classification performance than the sentence-augmented dataset.

The original dataset achieved a Macro F1-score of 0.9066, while the sentence-augmented dataset achieved a lower Macro F1-score of 0.8447.

These results indicate that sentence-level augmentation weakened overall class-balanced sentiment prediction performance across the Negative, Neutral, and Positive sentiment classes.

The lower Macro F1-score suggests that splitting reviews into isolated sentence-level samples may have reduced contextual consistency and introduced additional label noise, particularly for reviews containing mixed or nuanced sentiment expressions.

Overall, the chart further supports the conclusion that preserving the original review-level contextual structure produced more reliable multilingual sentiment classification performance than sentence-level augmentation.

## 21.7 English-Only vs English and Translated Malay Reviews

This experiment compares:

1. English-only review dataset
2. Combined English and translated Malay dataset

The objective is to determine whether retaining translated Malay reviews improves multilingual sentiment classification performance.

In [ ]:
# English-Only vs English and Translated Malay Reviews



# Create English-Only Dataset
english_only_df = df[
    df["Language"] == "English"
].copy()



# Create Combined English + Malay Dataset
english_malay_df = df[
    df["Language"].isin(["English", "Malay"])
].copy()



# Display Dataset Sizes
print(
    "English-only dataset size:",
    english_only_df.shape
)

print(
    "English and translated Malay dataset size:",
    english_malay_df.shape
)



# Display Sentiment Distribution
print("\nEnglish-only sentiment distribution:")

print(
    english_only_df["Sentiment"]
    .value_counts()
)


print(
    "\nEnglish and translated Malay sentiment distribution:"
)

print(
    english_malay_df["Sentiment"]
    .value_counts()
)

## Insights

The English-only dataset contains 436 reviews, while the combined English and translated Malay dataset contains 777 reviews.

Including translated Malay reviews substantially increased the number of available training samples across all sentiment classes, providing a larger multilingual dataset for recurrent neural network learning.

Both datasets maintain relatively balanced sentiment distributions, making them suitable for fair sentiment classification experiments without severe class imbalance issues.

This experiment investigates whether retaining translated Malay reviews improves multilingual sentiment classification performance compared to using English-only reviews.

The results will help determine whether multilingual data retention provides additional linguistic diversity and contextual variation that can improve deep learning model generalisation and sentiment learning capability.

## 21.8 Compare English-Only and Multilingual Datasets

This experiment evaluates:

1. English-only reviews
2. English + translated Malay reviews

The objective is to determine whether retaining translated Malay reviews improves sentiment classification performance.

Both datasets are evaluated using the same LSTM architecture and training workflow to ensure fair comparison.

In [ ]:
# Compare English-Only and Multilingual Datasets



# Train and Evaluate English-Only Dataset
english_only_result = train_and_evaluate_experiment(

    # English-only review dataset
    english_only_df,

    # Experiment name
    "English-Only Reviews"
)




# Train and Evaluate Combined English + Malay Dataset
english_malay_result = train_and_evaluate_experiment(

    # Combined multilingual dataset
    english_malay_df,

    # Experiment name
    "English and Translated Malay Reviews"
)

## Insights

The English-only dataset and the combined English and translated Malay dataset were evaluated using the same LSTM architecture and experimental workflow to ensure fair comparison.

The English-only dataset achieved an Accuracy of 0.9091 and a Macro F1-score of 0.9066, while the combined English and translated Malay dataset achieved a slightly higher Accuracy of 0.9231 and a higher Macro F1-score of 0.9155.

These results suggest that retaining translated Malay reviews improved overall multilingual sentiment classification performance in this project.

One possible reason is that the inclusion of translated Malay reviews increased dataset diversity and provided additional contextual sentiment patterns for the LSTM model to learn from. The larger multilingual dataset may have also improved the model’s ability to generalise across different review expressions and vocabulary usage.

The Positive sentiment class continued to achieve the strongest performance across both experiments, while the combined multilingual dataset also demonstrated improved balanced classification performance for the Negative and Neutral sentiment classes.

Overall, this experiment suggests that retaining translated Malay reviews was beneficial for sentiment classification performance, indicating that multilingual data enrichment can improve deep learning model generalisation when translation quality remains reasonably consistent.

## 21.9 Language Experiment Comparison

This section summarises the performance of:

1. English-only review dataset
2. English + translated Malay review dataset

The comparison focuses on:
- Accuracy
- Macro F1-score

This helps determine whether retaining translated Malay reviews improves multilingual sentiment classification performance.

In [ ]:
# Language Experiment Comparison



# Create Language Comparison Table
language_comparison_df = pd.DataFrame([

    {
        # English-only experiment
        "Experiment":
            english_only_result["Experiment"],

        # Testing accuracy
        "Accuracy":
            english_only_result["Accuracy"],

        # Macro-average F1-score
        "Macro F1-score":
            english_only_result["Macro F1-score"]
    },

    {
        # English + translated Malay experiment
        "Experiment":
            english_malay_result["Experiment"],

        # Testing accuracy
        "Accuracy":
            english_malay_result["Accuracy"],

        # Macro-average F1-score
        "Macro F1-score":
            english_malay_result["Macro F1-score"]
    }
])



# Round Numerical Values for Cleaner Presentation
language_comparison_df["Accuracy"] = (

    language_comparison_df["Accuracy"]
    .round(4)
)

language_comparison_df["Macro F1-score"] = (

    language_comparison_df["Macro F1-score"]
    .round(4)
)



# Display Comparison Table
language_comparison_df

## Insights

The language comparison table summarises the performance differences between the English-only dataset and the combined English and translated Malay dataset.

The combined English and translated Malay dataset achieved slightly higher Accuracy and Macro F1-score than the English-only dataset, indicating that retaining translated Malay reviews improved multilingual sentiment classification performance in this project.

Although the multilingual dataset introduced additional linguistic variation, the larger dataset size and increased diversity of sentiment expressions likely helped the LSTM model learn richer contextual patterns and improve generalisation capability.

The higher Macro F1-score also suggests that the multilingual dataset improved balanced classification performance across the Negative, Neutral, and Positive sentiment classes.

These findings indicate that retaining translated Malay reviews was beneficial for sentiment classification performance, suggesting that multilingual data enrichment can strengthen deep learning performance when translation quality remains sufficiently consistent.

Overall, the experiment demonstrates that increasing linguistic diversity through translated multilingual reviews improved overall sentiment classification capability within this project.

## 21.10 Visualise Accuracy Comparison for Language Experiment

This chart visually compares the testing accuracy of:

1. English-only review dataset
2. English + translated Malay review dataset

The objective is to determine whether retaining translated Malay reviews improves multilingual sentiment classification performance.

In [ ]:
# Visualise Accuracy Comparison for Language Experiment



# Create Figure
plt.figure(figsize=(8,5))



# Create Accuracy Comparison Bar Chart
ax = sns.barplot(

    # Experiment names on x-axis
    x="Experiment",

    # Accuracy values on y-axis
    y="Accuracy",

    # Language experiment comparison dataframe
    data=language_comparison_df
)



# Display Accuracy Values Above Bars
for i in ax.containers:

    ax.bar_label(
        i,
        fmt="%.4f"
    )



# Chart Title and Labels
plt.title(
    "Accuracy Comparison for Language Experiment"
)

plt.ylabel("Accuracy")

plt.xticks(rotation=0)


# Makes differences easier to visualise
plt.ylim(0.85, 0.95)



# Display Chart
plt.show()

## Insights

The accuracy comparison chart visually shows that the combined English and translated Malay dataset achieved slightly higher sentiment classification accuracy than the English-only dataset.

The English-only dataset achieved an accuracy of 0.9091, while the combined English and translated Malay dataset achieved a slightly higher accuracy of 0.9231.

These results suggest that retaining translated Malay reviews improved multilingual sentiment classification performance in this project.

Although multilingual datasets may introduce additional linguistic variation, the larger dataset size and increased diversity of sentiment expressions likely helped the LSTM model learn richer contextual patterns and improve generalisation capability.

The visualisation supports the conclusion that multilingual data retention was beneficial for sentiment classification performance, indicating that translated multilingual reviews can strengthen deep learning model learning when translation quality remains reasonably consistent.

## 21.11 Visualise Macro F1-score Comparison for Language Experiment

This chart visually compares the Macro F1-scores of:

1. English-only review dataset
2. English + translated Malay review dataset

Macro F1-score is important because it measures balanced sentiment classification performance across all classes.

This helps determine whether retaining translated Malay reviews improves multilingual sentiment classification performance.

In [ ]:

# Visualise Macro F1-score Comparison for Language Experiment




# Create Figure
plt.figure(figsize=(8,5))



# Create Macro F1-score Comparison Bar Chart
ax = sns.barplot(

    # Experiment names on x-axis
    x="Experiment",

    # Macro F1-score values on y-axis
    y="Macro F1-score",

    # Language comparison dataframe
    data=language_comparison_df
)



# Display Macro F1-score Values Above Bars
for i in ax.containers:

    ax.bar_label(
        i,
        fmt="%.4f"
    )



# Chart Title and Labels
plt.title(
    "Macro F1-score Comparison for Language Experiment"
)

plt.ylabel("Macro F1-score")

plt.xticks(rotation=0)


# Makes differences easier to visualise
plt.ylim(0.85, 0.95)



# Display Chart
plt.show()

## Insights

The Macro F1-score comparison chart visually shows that the combined English and translated Malay dataset achieved slightly stronger balanced sentiment classification performance than the English-only dataset.

The English-only dataset achieved a Macro F1-score of 0.9066, while the combined English and translated Malay dataset achieved a slightly higher Macro F1-score of 0.9155.

This suggests that retaining translated Malay reviews improved overall class-balanced sentiment prediction performance across the Negative, Neutral, and Positive sentiment classes.

The higher Macro F1-score indicates that the multilingual dataset helped the LSTM model learn a broader range of contextual sentiment patterns and improved generalisation capability across different sentiment categories.

Although multilingual datasets may introduce additional linguistic variation, the increased dataset diversity and larger number of training samples appear to have benefited recurrent neural network learning in this project.

Overall, the chart further supports the conclusion that retaining translated Malay reviews improved multilingual sentiment classification performance and produced more balanced sentiment prediction capability.

## 21.12 Summary of Additional Experiment Findings

This table summarises the findings from:

1. Sentence-level augmentation experiment
2. Language retention experiment

Final preprocessing decisions are selected based on empirical evaluation metrics rather than assumptions.

In [ ]:
# Summary of Additional Experiment Findings



# Create Summary Table
additional_experiment_summary_df = pd.DataFrame([

    {
        # Augmentation experiment
        "Experiment":
            "Original vs Sentence-Augmented",

        # Better-performing dataset
        "Best Choice":
            "Original Dataset",

        # Reason for selection
        "Reason":
            "Higher Accuracy and Macro F1-score"
    },

    {
        # Language experiment
        "Experiment":
            "English-only vs English + Translated Malay",

        # Better-performing dataset
        "Best Choice":
            "English + Translated Malay Reviews",

        # Reason for selection
        "Reason":
            "Higher Accuracy and Macro F1-score"
    }
])



# Display Summary Table
additional_experiment_summary_df

## Insights

The summary table consolidates the findings from the additional preprocessing and data engineering experiments.

The original review dataset outperformed the sentence-augmented dataset, indicating that preserving full review-level contextual information was more beneficial than generating additional sentence-level samples through augmentation.

In contrast, the combined English and translated Malay dataset outperformed the English-only dataset, suggesting that retaining translated multilingual reviews improved sentiment classification performance by increasing linguistic diversity and training data coverage.

These findings demonstrate that dataset quality, contextual consistency, and meaningful linguistic diversity all play important roles in recurrent neural network learning performance.

The experiments also show that increasing dataset size alone does not necessarily improve performance, especially when augmentation introduces label inconsistency or weakens contextual coherence.

Overall, the final preprocessing decisions were selected based on empirical evaluation results rather than assumptions, improving the reliability, fairness, and validity of the multilingual sentiment classification workflow.

## 21.13 Final Language Retention Decision

The assignment specifically asks whether reviews in other languages should be retained or discarded.

Based on the language experiment, the combined English and translated Malay dataset achieved stronger performance than the English-only dataset.

The English-only dataset achieved an Accuracy of 0.9091 and a Macro F1-score of 0.9066, while the English and translated Malay dataset achieved a higher Accuracy of 0.9231 and a higher Macro F1-score of 0.9155.

Therefore, Malay reviews were retained after translation into English rather than being discarded. This decision was made because the translated Malay reviews increased the amount of available training data and provided additional sentiment expression patterns that improved overall model generalisation.

Very small language categories were discarded because they contained too few samples to support reliable training and fair evaluation.

Overall, the analysis shows that retaining translated Malay reviews was beneficial, while extremely small language groups were removed to maintain dataset quality and evaluation reliability.

# Step 23. Class Weight Experiment

Class weights are computed to reduce possible bias toward larger sentiment classes during model training.

Minority classes receive larger weights so that prediction errors on those classes contribute more strongly to the training loss.

This experiment helps evaluate whether balancing class importance improves sentiment classification performance.

In [ ]:
# Class Weight Experiment



# Compute Balanced Class Weights
class_weights = compute_class_weight(

    # Automatically balance classes
    class_weight="balanced",

    # Unique sentiment classes
    classes=np.unique(y_train),

    # Training labels
    y=y_train
)



# Convert Class Weights into Dictionary Format
class_weight_dict = dict(

    enumerate(class_weights)
)



# Display Computed Class Weights
print("Class Weights:")

print(class_weight_dict)

## Insights

The class weight experiment was conducted to reduce potential model bias toward larger sentiment classes during training.

Balanced class weights assign greater importance to minority classes by increasing the penalty for misclassifying underrepresented sentiment categories.

The computed class weights indicate that the Negative and Neutral sentiment classes received higher weights because they contained fewer training samples compared to the Positive class, which received the lowest weight due to its larger representation in the dataset.

The relatively moderate differences between class weights suggest that the dataset was not severely imbalanced, although some variation between sentiment categories still existed.

Overall, the class weight experiment helps evaluate whether balancing class importance can improve the model’s ability to classify minority sentiment categories more effectively and reduce prediction bias toward the majority Positive class.

## 23.1 Train Weighted LSTM Model

An LSTM model is trained using balanced class weights to test whether increasing the importance of minority sentiment classes improves classification performance.

The same LSTM architecture is used to ensure fair comparison with the earlier unweighted LSTM model.

In [ ]:
# Train Weighted LSTM Model



# Build Weighted LSTM Model
weighted_lstm_model = build_lstm_experiment_model()


# Train Weighted LSTM Model
weighted_history = weighted_lstm_model.fit(

    # Training input sequences
    X_train,

    # Training sentiment labels
    y_train,

    # Number of training epochs
    epochs=15,

    # Batch size
    batch_size=32,

    # Validation split
    validation_split=0.2,

    # Early stopping callback
    callbacks=[early_stop],

    # Apply balanced class weights
    class_weight=class_weight_dict
)

## Insights

The weighted LSTM model was trained using balanced class weights to evaluate whether increasing the importance of minority sentiment classes improves classification performance.

The training process demonstrated stable learning behaviour, with both training accuracy and validation accuracy improving steadily across epochs.

The weighted model achieved strong validation performance, reaching validation accuracy levels similar to the earlier unweighted LSTM model. This suggests that class weighting provided only limited additional improvement because the original dataset was already relatively balanced across sentiment categories.

The training curves also indicate that the weighted LSTM model successfully learned meaningful sequential sentiment patterns while maintaining relatively stable validation performance throughout training.

Although class weighting slightly increased the importance of the Negative and Neutral sentiment classes during optimisation, the overall performance gains remained modest. This suggests that the dataset did not suffer from severe class imbalance issues requiring aggressive balancing strategies.

Overall, the experiment demonstrates that balanced class weighting can help reduce potential prediction bias toward majority classes, although its effectiveness may be limited when class distributions are already reasonably balanced.

## 23.2 Evaluate Weighted LSTM Model

The weighted LSTM model is evaluated on unseen test data to determine whether applying class weights improved sentiment classification performance.

Test loss and test accuracy are compared against the earlier unweighted LSTM model.

In [ ]:

# Evaluate Weighted LSTM Model


# Evaluate Weighted LSTM Model on Test Set

weighted_loss, weighted_acc = (

    weighted_lstm_model.evaluate(

        # Testing input sequences
        X_test,

        # Testing sentiment labels
        y_test
    )
)



# Display Test Loss
print(

    f"Weighted LSTM Test Loss: "

    f"{weighted_loss:.4f}"
)



# Display Test Accuracy
print(

    f"Weighted LSTM Test Accuracy: "

    f"{weighted_acc:.4f}"
)

## Insights

The weighted LSTM model achieved a test accuracy of approximately 91.0% and a test loss of approximately 0.2444 on unseen review data.

Compared to the earlier unweighted LSTM model, the weighted model achieved slightly higher testing accuracy and lower testing loss. This suggests that applying balanced class weights helped improve overall sentiment classification performance for this dataset.

The improvement may be due to the increased importance assigned to the Negative and Neutral sentiment classes during training, allowing the model to learn minority sentiment patterns more effectively and reduce prediction bias toward the majority Positive class.

Although the dataset was already relatively balanced, the experiment demonstrates that moderate class weighting can still provide performance benefits by improving class-sensitive learning behaviour during optimisation.

The weighted LSTM model maintained strong generalisation capability while achieving more balanced sentiment classification performance across the three sentiment classes.

Overall, the experiment suggests that applying balanced class weights slightly improved the effectiveness of the LSTM sentiment classification model for this multilingual pizza review dataset.

## 23.3 Weighted LSTM Classification Report

The weighted LSTM model is evaluated using:

- Precision
- Recall
- F1-score

for each sentiment class:

- Negative
- Neutral
- Positive

This helps determine whether class weighting improved balanced sentiment classification performance across all classes.

In [ ]:

# Weighted LSTM Classification Report



# Generate Prediction Probabilities
weighted_pred_prob = (

    weighted_lstm_model.predict(
        X_test
    )
)



# Convert Probabilities into Predicted Classes
weighted_pred_classes = np.argmax(

    weighted_pred_prob,

    axis=1
)



# Display Classification Report
print(

    classification_report(

        y_test,

        weighted_pred_classes,

        target_names=encoder.classes_
    )
)

## Insights

The weighted LSTM classification report shows that the model achieved strong and well-balanced performance across the Negative, Neutral, and Positive sentiment classes.

The weighted model achieved high precision, recall, and F1-scores for all sentiment categories, indicating that applying balanced class weights helped the LSTM model learn minority sentiment patterns more effectively during training.

The Negative sentiment class achieved particularly strong recall performance of approximately 91%, suggesting that class weighting helped the model identify negative reviews more effectively and reduced bias toward the majority Positive sentiment class.

The Neutral sentiment class also achieved strong precision and F1-score values, indicating stable performance when handling more subtle or ambiguous sentiment expressions.

The Positive sentiment class continued to achieve the strongest overall performance because strongly positive wording was generally easier for the LSTM model to classify consistently.

Compared to the earlier unweighted LSTM model, the weighted model demonstrated slightly improved balanced classification performance while maintaining strong overall accuracy and generalisation capability.

Overall, the classification report suggests that balanced class weighting contributed positively to multilingual sentiment classification performance by encouraging more balanced learning behaviour across all sentiment classes.

## 23.4 Class Weight Experiment Comparison

This comparison summarises the performance of:

- Original unweighted LSTM model
- Weighted LSTM model

The objective is to determine whether applying balanced class weights improved overall sentiment classification performance.

In [ ]:

# Class Weight Experiment Comparison



# Create Comparison DataFrame

class_weight_comparison_df = pd.DataFrame([

    {
        # Original LSTM model
        "Model": "Original LSTM",

        # Test accuracy
        "Accuracy": acc2
    },

    {
        # Weighted LSTM model
        "Model": "Weighted LSTM",

        # Test accuracy
        "Accuracy": weighted_acc
    }
])



# Round Accuracy Values

class_weight_comparison_df["Accuracy"] = (

    class_weight_comparison_df["Accuracy"]

    .round(4)
)



# Display Comparison Table

class_weight_comparison_df

## Insights

The class weight comparison table shows that the weighted LSTM model achieved slightly higher overall accuracy than the original unweighted LSTM model.

The original LSTM model achieved an accuracy of approximately 90.4%, while the weighted LSTM model achieved a slightly improved accuracy of approximately 91.0%.

These results suggest that applying balanced class weights contributed positively to sentiment classification performance for this dataset.

One possible reason is that class weighting increased the importance of the Negative and Neutral sentiment classes during optimisation, helping the model reduce prediction bias toward the majority Positive class and improve balanced learning behaviour.

Although the dataset was already relatively balanced, the experiment demonstrates that moderate class weighting can still provide small but meaningful improvements in model generalisation and class-sensitive prediction performance.

Overall, the weighted LSTM model achieved the strongest overall sentiment classification performance among the class weight experiment configurations.

## 23.5 Save Final Weighted LSTM Model

After evaluating the class weight experiment, the weighted LSTM model achieved slightly stronger overall sentiment classification performance than the earlier unweighted LSTM model.

Therefore, the weighted LSTM model is selected as the final best-performing model for this project.

The final weighted LSTM model is saved as an .h5 file for:

- reproducibility
- future reuse
- deployment purposes
- submission requirements

Saving the trained model allows future predictions to be generated without retraining the network.

saves:

the ENTIRE MODEL.

Including:

architecture
weights
optimizer state

In [ ]:
# =========================================================
# Save Final Weighted LSTM Model
# =========================================================

weighted_lstm_model.save(
    "best_weighted_lstm_model.h5"
)

# Verify Model File Exists
print(
    os.path.exists(
        "best_weighted_lstm_model.h5"
    )
)

# Display Saved File Name
print(
    "Saved final model file: "
    "best_weighted_lstm_model.h5"
)

## Insights

The final weighted LSTM model was successfully saved as an .h5 file.

Saving the trained model improves reproducibility and allows the model to be reused later without requiring retraining.

The saved model can also be used for future prediction tasks, deployment, or additional evaluation experiments.

The weighted LSTM model was selected as the final best-performing model because it achieved slightly stronger overall sentiment classification performance than the earlier unweighted LSTM model.

# Step 24. Final Weighted LSTM Model Detailed Evaluation

The final selected weighted LSTM model is evaluated using precision, recall, F1-score, and confusion matrix analysis.

These metrics provide detailed insight into how effectively the weighted LSTM model classified the Negative, Neutral, and Positive sentiment classes.

In [ ]:

# Final Weighted LSTM Model Detailed Evaluation

# Generate Prediction Probabilities
weighted_lstm_pred_prob = weighted_lstm_model.predict(
    X_test
)

# Convert Probabilities into Predicted Classes
weighted_lstm_pred_classes = np.argmax(
    weighted_lstm_pred_prob,
    axis=1
)

# Display Classification Report
print(

    classification_report(

        y_test,

        weighted_lstm_pred_classes,

        target_names=encoder.classes_
    )
)

## Insights

The final weighted LSTM model achieved strong classification performance across the Negative, Neutral, and Positive sentiment classes.

The model achieved an overall accuracy of approximately 91%, with a macro average F1-score of 0.90 and a weighted average F1-score of 0.91. This indicates that the model performed well overall while maintaining relatively balanced performance across all sentiment categories.

The Negative class achieved a recall of 0.91, suggesting that class weighting helped the model identify negative reviews more effectively. This is important because negative reviews can be more difficult to classify when their wording is subtle or context-dependent.

The Neutral class achieved strong precision of 0.93 and an F1-score of 0.89, showing that the model performed well when identifying moderate or less emotionally intense reviews.

The Positive class remained the strongest overall category, achieving an F1-score of 0.95. This suggests that strongly positive sentiment expressions were generally easier for the model to recognise.

Overall, the final weighted LSTM model demonstrated strong generalisation capability and balanced multilingual sentiment classification performance on unseen review data.

## 24.1 Final Weighted LSTM Confusion Matrix

A confusion matrix is generated to visualise how the final weighted LSTM model classified:

- Negative reviews
- Neutral reviews
- Positive reviews

The confusion matrix helps identify:

- correctly classified reviews
- misclassified reviews
- which sentiment classes are more difficult for the model to distinguish

In [ ]:

# Final Weighted LSTM Confusion Matrix

# Generate Weighted LSTM Confusion Matrix
weighted_lstm_cm = confusion_matrix(
    y_test,
    weighted_lstm_pred_classes
)

# Plot Confusion Matrix
plt.figure(figsize=(6,4))

sns.heatmap(
    weighted_lstm_cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=encoder.classes_,
    yticklabels=encoder.classes_
)

# Configure Plot Labels
plt.title("Final Weighted LSTM Confusion Matrix")
plt.xlabel("Predicted Sentiment")
plt.ylabel("Actual Sentiment")

plt.show()

## Insights

The confusion matrix visualises how effectively the final weighted LSTM model classified each sentiment category.

Most reviews were correctly classified along the diagonal of the matrix, indicating strong overall sentiment classification performance across all sentiment classes.

The Negative sentiment class achieved strong classification performance, with 40 out of 44 Negative reviews correctly classified. This suggests that class weighting helped improve the model’s ability to recognise minority or more difficult sentiment categories.

The Neutral sentiment class remained slightly more challenging, with some Neutral reviews being misclassified as Negative sentiment. This is likely because Neutral reviews often contain subtle, moderate, or ambiguous emotional wording that overlaps with mildly negative expressions.

The Positive sentiment class achieved the strongest performance overall, with 64 out of 68 Positive reviews correctly classified. Strongly positive reviews generally contain clearer sentiment cues, making them easier for the weighted LSTM model to identify.

Compared to the earlier unweighted LSTM model, the weighted LSTM confusion matrix demonstrates slightly improved class-balanced prediction behaviour, particularly for the Negative sentiment category.

Overall, the confusion matrix demonstrates that the final weighted LSTM model achieved strong and relatively balanced multilingual sentiment classification performance while still facing some difficulty with nuanced or ambiguous sentiment expressions.

# Step 25. Error Analysis of Misclassified Reviews

Misclassified reviews are analysed to identify:

- model weaknesses
- ambiguous sentiment patterns
- translation-related issues
- confidence behaviour

Error analysis helps explain why certain reviews were incorrectly classified by the final weighted LSTM model.

In [ ]:

# Error Analysis of Misclassified Reviews

# Create Error Analysis DataFrame
error_analysis_df = df.loc[test_indices].copy()



# Add Actual Sentiment Labels
error_analysis_df["Actual Sentiment"] = (

    encoder.inverse_transform(y_test)
)



# Add Predicted Sentiment Labels
error_analysis_df["Predicted Sentiment"] = (

    encoder.inverse_transform(weighted_lstm_pred_classes)
)



# Add Prediction Confidence Scores
error_analysis_df["Prediction Confidence"] = (

    np.max(weighted_lstm_pred_prob, axis=1)
)



# Filter Misclassified Reviews
misclassified_df = error_analysis_df[

    error_analysis_df["Actual Sentiment"]

    !=

    error_analysis_df["Predicted Sentiment"]
]



# Display Sample Misclassified Reviews
misclassified_df[

    [
        "Review",
        "Review_Translated",
        "Language",
        "Score",
        "Actual Sentiment",
        "Predicted Sentiment",
        "Prediction Confidence"
    ]

].head(10)

## Insights

The misclassified review examples provide deeper insight into the limitations and challenges faced by the final weighted LSTM sentiment classification model.

Several prediction errors occurred in reviews containing ambiguous wording, short text, mixed sentiment expressions, or inconsistencies between textual sentiment and numerical review scores.

For example, reviews such as “Somewhat bland.” and “Crust was thin and crispy, just right.” contain moderate or less emotionally intense wording, making them harder to classify clearly as Negative, Neutral, or Positive.

Some reviews were also misclassified despite containing clear sentiment cues. For instance, “So good!” was labelled as Positive but predicted as Negative. This suggests that the model may occasionally learn misleading keyword associations or overfit to noisy sentiment patterns in the dataset.

Several Malay reviews translated into English were also misclassified, such as “I don't like the taste of sambal and mayo.” being predicted as Positive despite clearly negative wording. This suggests that translation effects, food-related vocabulary, or learned keyword associations may have influenced the prediction incorrectly.

The error analysis also shows possible label noise. For example, “The sambal sauce made me sweat, cry, and smile at the same time!” was labelled as Negative due to its numerical score, even though the text contains mixed or partially positive emotional expressions. Such inconsistencies can make supervised learning more difficult.

Interestingly, some incorrect predictions were made with high confidence scores, indicating that the weighted LSTM model was occasionally overconfident when handling misleading keywords, ambiguous sentiment, or noisy labels.

Overall, the error analysis shows that the final weighted LSTM model learned many meaningful multilingual sentiment patterns effectively, but still faced challenges involving ambiguity, translation consistency, short-text interpretation, mixed sentiment, and label noise.

## 25.1 Misclassification Rate Analysis

The misclassification rate measures how frequently the final weighted LSTM model predicted the wrong sentiment label on unseen test data.

This provides an additional summary metric for evaluating model reliability and generalisation.

In [ ]:

# Misclassification Rate Analysis

# Display Total Number of Test Samples
print(
    f"Total test samples: {len(error_analysis_df)}"
)

# Display Number of Misclassified Samples
print(
    f"Number of misclassified samples: "
    f"{len(misclassified_df)}"
)

# Calculate and Display Misclassification Rate
print(

    f"Misclassification rate: "

    f"{len(misclassified_df) / len(error_analysis_df):.4f}"
)

## Insights

The misclassification analysis shows that the final weighted LSTM model incorrectly classified 14 out of 156 unseen test reviews, resulting in a misclassification rate of approximately 0.0897.

This means that around 91% of the test reviews were classified correctly, which is consistent with the strong overall testing accuracy achieved by the final weighted LSTM model.

The relatively low misclassification rate indicates that the model generalised effectively on unseen multilingual pizza review data.

However, the remaining prediction errors show that the model still faced challenges when handling ambiguous wording, short reviews, mixed sentiment expressions, translation-related inconsistencies, and noisy sentiment labels.

Overall, the final weighted LSTM model demonstrated strong reliability and generalisation performance, while still showing some limitations when interpreting nuanced or context-dependent sentiment expressions.

## 25.2 Misclassification Distribution by Sentiment Class

This analysis identifies which actual sentiment classes contributed most to the final weighted LSTM model's prediction errors.

Understanding class-level error distribution helps:

- identify weaker sentiment categories
- analyse model limitations
- support the final evaluation discussion

In [ ]:
# Misclassification Distribution by Sentiment Class

# Count Misclassified Reviews by Actual Sentiment
misclassification_counts = (

    misclassified_df["Actual Sentiment"]

    .value_counts()
)

# Display Misclassification Counts
print(
    "Misclassified Reviews by Actual Sentiment:"
)

print(misclassification_counts)

## Insights

The misclassification distribution analysis shows how prediction errors were distributed across the actual sentiment classes.

The Neutral sentiment class contributed the highest number of misclassified reviews, with 6 errors, while the Negative and Positive classes each contributed 4 errors.

This suggests that Neutral reviews remained the most challenging category for the final weighted LSTM model to classify accurately. Neutral sentiment often contains more subtle, moderate, or mixed emotional wording, making it harder to distinguish from nearby Negative or Positive sentiment classes.

The Negative and Positive classes produced fewer errors, indicating that the model was generally more effective at identifying stronger emotional expressions.

The relatively balanced distribution of errors also suggests that the final weighted LSTM model did not fail heavily on any single sentiment category.

Overall, the analysis shows that the weighted LSTM model achieved strong class-balanced performance while still facing some difficulty with more ambiguous Neutral reviews.

# Step 25.3 Final Recurrent Neural Network Architecture Comparison

Multiple recurrent neural network architectures were evaluated throughout this project to determine which model produced the strongest multilingual sentiment classification performance.

The architectures differed in sequential learning capability, computational complexity, memory handling, and contextual learning behaviour.

The comparison below summarises the key characteristics, strengths, and limitations of each architecture investigated in this project.

In [ ]:
# Final Recurrent Neural Network Architecture Comparison

architecture_comparison_df = pd.DataFrame([

    {
        "Architecture": "SimpleRNN",

        "Strengths":
            "Simple sequential baseline model",

        "Limitations":
            "Weak long-term contextual memory",

        "Performance Summary":
            "Lowest overall performance"
    },

    {
        "Architecture": "LSTM",

        "Strengths":
            "Strong long-term contextual learning",

        "Limitations":
            "Higher computational complexity",

        "Performance Summary":
            "Strong sentiment classification performance"
    },

    {
        "Architecture": "GRU",

        "Strengths":
            "Efficient sequential learning with fewer parameters",

        "Limitations":
            "Slightly simpler memory mechanism",

        "Performance Summary":
            "Strong performance with faster training"
    },

    {
        "Architecture": "Weighted LSTM",

        "Strengths":
            "Improved balanced sentiment learning",

        "Limitations":
            "Slightly increased training complexity",

        "Performance Summary":
            "Best overall classification performance"
    }
])



# Display Architecture Comparison Table
architecture_comparison_df

## Insights

The architecture comparison summarises the strengths, limitations, and overall performance of the recurrent neural network models evaluated throughout this project.

SimpleRNN provided a useful baseline model but demonstrated weaker long-term contextual learning capability compared to the more advanced recurrent architectures.

The standard LSTM model achieved strong sentiment classification performance by effectively learning long-term sequential dependencies within multilingual review text.

GRU also demonstrated strong performance while using a simpler gating structure and fewer parameters, making it computationally efficient for sequential sentiment analysis tasks.

The weighted LSTM model ultimately achieved the strongest overall sentiment classification performance by combining effective long-term contextual learning with improved balanced sentiment classification across the Negative, Neutral, and Positive classes.

Overall, the comparative experiments demonstrated that more advanced recurrent architectures with stronger contextual memory mechanisms produced more reliable multilingual sentiment classification performance than simpler recurrent models.

# Step 26. Final Decision Justification

The final modelling decisions were selected based on experimental evaluation rather than assumptions.

Classification was selected instead of regression because the objective of the project was to categorise pizza reviews into three clearly defined sentiment classes: Negative, Neutral, and Positive.

Malay reviews were translated into English to investigate whether combining multilingual reviews into a unified language representation could improve sentiment learning consistency across the dataset.

Very small language categories were removed because they contained insufficient samples to support reliable training and fair evaluation.

Sentence-level augmentation was also investigated because the dataset size was relatively limited. However, experimental results showed that preserving complete review-level context produced stronger sentiment classification performance than splitting reviews into isolated sentence-level samples.

Several recurrent neural network architectures were evaluated throughout the project, including a Baseline SimpleRNN model, an LSTM model, and a GRU model. 

Experimental comparisons showed that the LSTM architecture achieved stronger and more stable sentiment classification performance than the other recurrent neural network architectures. The LSTM model demonstrated effective capability in learning complex sequential sentiment patterns and longer contextual dependencies within multilingual review text.

Class weighting was additionally explored to evaluate whether balancing sentiment class importance could improve minority sentiment classification performance. Experimental results showed that the weighted LSTM model achieved slightly improved overall performance and more balanced classification behaviour across the Negative, Neutral, and Positive sentiment classes.

Additional preprocessing experiments also demonstrated that preserving contextual consistency and label quality contributed more effectively to sentiment classification performance than simply increasing dataset size through sentence-level augmentation or multilingual translation.

The final weighted LSTM model was therefore selected as the best-performing model because it achieved strong overall accuracy, balanced Macro F1-score performance, effective generalisation on unseen test data, and relatively balanced classification capability across all sentiment categories.

Overall, the final solution was selected using controlled experiments, fair evaluation procedures, multiple performance metrics, comparative experimentation, and detailed error analysis to ensure reliable and well-justified sentiment classification performance.

## Justification for Selecting LSTM Instead of Bidirectional LSTM (BiLSTM)

Bidirectional LSTM (BiLSTM) architectures were considered during the model planning stage because they are capable of learning contextual information from both forward and backward directions within a text sequence. This can potentially improve contextual understanding and sentiment interpretation for complex natural language processing tasks.

However, the standard weighted LSTM architecture was ultimately selected as the final model because it was more suitable for the characteristics and constraints of this project dataset.

First, the multilingual pizza review dataset was relatively limited in size, containing fewer than one thousand usable reviews after preprocessing and filtering. More complex architectures such as BiLSTM contain additional trainable parameters and higher computational complexity, which may increase the risk of overfitting when training on smaller datasets.

Second, many reviews in the dataset were relatively short and contained simple sentence structures such as “So good!”, “Somewhat bland.”, and “Pizza lacks flavor.” Since the reviews generally did not contain very long contextual dependencies, the additional backward contextual learning capability provided by BiLSTM was unlikely to produce substantial performance improvement.

Third, the weighted LSTM model already demonstrated strong sentiment classification performance, achieving high Accuracy, strong Macro F1-score, balanced class-level performance, and relatively low misclassification rates across the Negative, Neutral, and Positive sentiment classes. This indicates that the standard LSTM architecture was already sufficiently capable of learning meaningful sequential sentiment patterns from the multilingual review data.

In addition, the weighted LSTM architecture provided a good balance between:
- contextual learning capability
- computational efficiency
- model complexity
- training stability
- generalisation performance

The project also focused heavily on controlled experimentation, preprocessing evaluation, multilingual analysis, class weighting, augmentation experiments, and detailed error analysis rather than relying solely on increasingly complex neural network architectures.

Overall, the weighted LSTM model was considered the more appropriate and practically suitable architecture for this project because it achieved strong and reliable sentiment classification performance while maintaining lower complexity and reduced overfitting risk compared to a more computationally intensive BiLSTM architecture.

# Step 27. Limitations

Several limitations were identified throughout this sentiment classification project.

First, the dataset size was relatively limited, meaning that model performance may vary depending on train-test split configuration and random seed selection. A larger dataset would likely improve model generalisation, robustness, and training stability.

Second, some reviews were extremely short or contained limited contextual information, making sentiment interpretation more difficult for recurrent neural network models. Ambiguous reviews such as “Just weird.” or “Somewhat bland.” provided insufficient semantic context for reliable classification.

Third, Malay-to-English translation may have introduced translation inconsistencies or caused subtle emotional meaning and contextual nuance to be partially lost during preprocessing. Although the combined multilingual dataset achieved strong overall performance, translation-related semantic variation may still have contributed to certain multilingual misclassifications.

Fourth, sentence-level augmentation introduced potential label noise because individual sentences inherited the sentiment label of the original full review even when certain sentences expressed weaker, mixed, or different sentiment polarity independently.

Finally, some reviews contained inconsistencies between textual sentiment and numerical review scores, which may have introduced noisy labels during supervised learning and contributed to certain high-confidence misclassifications.

Despite these limitations, the final weighted LSTM model was still able to achieve strong and relatively balanced sentiment classification performance across the Negative, Neutral, and Positive sentiment classes.

# Step 28. Future Improvements

Several future improvements could further enhance the performance, robustness, and generalisation capability of the sentiment classification system.

First, more advanced recurrent neural network architectures such as Bidirectional LSTM (BiLSTM) or Bidirectional GRU (BiGRU) models could be explored in future studies to investigate whether bidirectional contextual learning improves sentiment classification performance on larger or more complex multilingual review datasets.

Although the final weighted LSTM model already demonstrated strong sequential learning capability and generalisation performance for this project dataset, future research may evaluate whether bidirectional architectures provide additional benefits under different dataset characteristics or larger-scale sentiment analysis scenarios.

Second, pretrained word embeddings such as Word2Vec, GloVe, or FastText could be incorporated to provide richer semantic representations compared to randomly initialised embeddings. This may improve the model’s ability to capture semantic similarity, contextual relationships, and multilingual language patterns more effectively.

Third, multilingual sentiment learning could potentially be improved using higher-quality Malay-to-English translation systems or manually verified translations to reduce semantic inconsistency and emotional nuance loss introduced during preprocessing.

More sophisticated context-aware data augmentation techniques could also be investigated to generate additional training samples while preserving sentiment consistency and reducing label noise.

In addition, systematic hyperparameter tuning could be performed to optimise learning rate, dropout rate, recurrent unit size, embedding dimensions, batch size, and sequence length for improved model performance and training stability.

Attention mechanisms may also help the model focus more effectively on sentiment-related keywords and important contextual phrases within reviews, improving interpretability and contextual learning behaviour.

Future studies could also evaluate the models using cross-validation and larger real-world multilingual review datasets to improve robustness, reduce performance variability, and provide more reliable estimates of model generalisation capability.

Overall, these improvements may further enhance classification accuracy, contextual understanding, multilingual robustness, training stability, and overall model generalisation performance.

# Step 29. Conclusion

In this project, recurrent neural network models were successfully applied to multilingual pizza review sentiment analysis.

The dataset was cleaned, filtered, translated, tokenized, padded, and split into training and testing sets. Malay reviews were translated into English to investigate whether combining multilingual reviews into a unified language representation could improve sentiment learning consistency.

A Baseline SimpleRNN model was first developed to establish a benchmark for sentiment classification performance. More advanced LSTM and GRU architectures were then developed and comparatively evaluated to improve multilingual sentiment classification capability.

The models were evaluated using multiple performance metrics and evaluation techniques including accuracy, loss, confusion matrices, training curves, sample predictions, and detailed error analysis.

Additional experiments were also conducted to investigate sentence-level augmentation, multilingual language handling, and class weighting strategies. These experiments helped justify the final preprocessing and modelling decisions using empirical evaluation results rather than assumptions.

Experimental results showed that the LSTM architecture achieved the strongest and most stable overall sentiment classification performance among the evaluated recurrent neural network models. The weighted LSTM model further improved balanced sentiment classification behaviour by slightly enhancing minority sentiment prediction capability and reducing overall misclassification rate.

The project also demonstrated that preserving contextual consistency and label quality contributed more effectively to sentiment classification performance than simply increasing dataset size through sentence-level augmentation or multilingual translation.

Overall, this project demonstrates how recurrent neural network architectures such as LSTM and GRU can be effectively applied to multilingual sentiment classification tasks, while also highlighting the importance of evidence-based preprocessing, controlled experimentation, detailed evaluation, and critical error analysis in deep learning workflows.